In [32]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU Colab indisponible. Va dans Runtime > Change runtime type > T4 GPU, "
        "puis redémarre la session avant de relancer ce notebook."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")

PyTorch version: 2.14.1+cpu
CUDA available: False
Aucun GPU disponible


In [33]:
!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    evaluate \
    sentencepiece

In [34]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

PyTorch: 2.14.1+cpu
Transformers: 5.22.1
Datasets: 4.8.5
PEFT: 0.21.2
TRL: 1.17.1
BitsAndBytes: 0.50.2


In [37]:
# ============================================================
# DATA SOURCE CATALOG
# ============================================================

sources = [
    {
        "source_name": "Atlaset",
        "source_type": "HuggingFace Dataset",
        "url": "https://huggingface.co/datasets/atlasia/Atlaset",
        "data_type": "Moroccan Darija corpus",
        "status": "TO_EXPLORE"
    },
    {
        "source_name": "DODa",
        "source_type": "GitHub Dataset",
        "url": "https://github.com/darija-open-dataset/dataset",
        "data_type": "Darija parallel and conversational data",
        "status": "TO_EXPLORE"
    }
]

catalog = pd.DataFrame(sources)

catalog.to_csv(
    PROJECT_DIR / "reports" / "source_catalog.csv",
    index=False
)

display(catalog)

,source_name,source_type,url,data_type,status
0,Atlaset,HuggingFace Dataset,https://huggingface.co/datasets/atlasia/Atlaset,Moroccan Darija corpus,TO_EXPLORE
1,DODa,GitHub Dataset,https://github.com/darija-open-dataset/dataset,Darija parallel and conversational data,TO_EXPLORE


In [40]:
# ==========================================
# PROJECT: MOROCCAN DARIJA LLM
# PHASE 1: DATA ENGINEERING
# ==========================================

import os
from pathlib import Path

PROJECT_DIR = Path("/content/darija_llm_project")

folders = [
    "data/bronze",
    "data/silver",
    "data/gold",
    "notebooks",
    "reports",
    "models",
    "logs"
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("Project structure created!")

for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_dir():
        print("📁", path.relative_to(PROJECT_DIR))

Project structure created!
📁 data
📁 data/bronze
📁 data/gold
📁 data/silver
📁 logs
📁 models
📁 notebooks
📁 reports


In [43]:
# ============================================================
# MOROCCAN DARIJA LLM
# END-TO-END DATA ENGINEERING & FINE-TUNING
# ============================================================

import os
import json
import hashlib
import pandas as pd
import requests
import torch
from pathlib import Path
from datetime import datetime

# Project configuration
PROJECT_NAME = "Moroccan_Darija_LLM"
PROJECT_DIR = Path("/content") / PROJECT_NAME

# Data Lake structure
FOLDERS = [
    "data/bronze",
    "data/silver",
    "data/gold",
    "data/external",
    "reports",
    "models",
    "logs"
]

for folder in FOLDERS:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

# Environment information
print("=" * 55)
print("PROJECT:", PROJECT_NAME)
print("DATE:", datetime.now())
print("PYTORCH:", torch.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("GPU: None (Running on CPU)")
print("=" * 55)

print("\nPROJECT STRUCTURE")

for path in sorted(PROJECT_DIR.rglob("*")):
    if path.is_dir():
        print("📁", path.relative_to(PROJECT_DIR))

PROJECT: Moroccan_Darija_LLM
DATE: 2026-11-08 22:33:27.396486
PYTORCH: 2.14.1+cpu
GPU: None (Running on CPU)

PROJECT STRUCTURE
📁 data
📁 data/bronze
📁 data/external
📁 data/gold
📁 data/silver
📁 logs
📁 models
📁 reports


In [44]:
# ============================================================
# DATA SOURCE CATALOG
# ============================================================

sources = [
    {
        "source_name": "Atlaset",
        "source_type": "HuggingFace Dataset",
        "url": "https://huggingface.co/datasets/atlasia/Atlaset",
        "data_type": "Moroccan Darija corpus",
        "status": "TO_EXPLORE"
    },
    {
        "source_name": "DODa",
        "source_type": "GitHub Dataset",
        "url": "https://github.com/darija-open-dataset/dataset",
        "data_type": "Darija parallel and conversational data",
        "status": "TO_EXPLORE"
    }
]

catalog = pd.DataFrame(sources)

catalog.to_csv(
    PROJECT_DIR / "reports" / "source_catalog.csv",
    index=False
)

display(catalog)

,source_name,source_type,url,data_type,status
0,Atlaset,HuggingFace Dataset,https://huggingface.co/datasets/atlasia/Atlaset,Moroccan Darija corpus,TO_EXPLORE
1,DODa,GitHub Dataset,https://github.com/darija-open-dataset/dataset,Darija parallel and conversational data,TO_EXPLORE


In [45]:
# ============================================================
# EXTRACT: DATASET METADATA
# ============================================================

from huggingface_hub import HfApi

api = HfApi()

dataset_ids = [
    "atlasia/Atlaset"
]

for dataset_id in dataset_ids:

    print("\n" + "=" * 60)
    print("DATASET:", dataset_id)

    try:
        info = api.dataset_info(dataset_id)

        print("Author:", info.author)
        print("Last modified:", info.last_modified)
        print("Tags:", info.tags[:10] if info.tags else None)

        print("\nAvailable files:")

        files = api.list_repo_files(
            repo_id=dataset_id,
            repo_type="dataset"
        )

        for file in files[:20]:
            print("-", file)

    except Exception as e:
        print("Error:", str(e))


DATASET: atlasia/Atlaset
Author: atlasia
Last modified: 2025-03-31 23:04:58+00:00
Tags: ['language:ary', 'language:ar', 'size_categories:1M<n<10M', 'format:parquet', 'modality:tabular', 'modality:text', 'library:datasets', 'library:dask', 'library:mlcroissant', 'library:polars']

Available files:
- .gitattributes
- README.md
- data/test-00000-of-00001.parquet
- data/train-00000-of-00005.parquet
- data/train-00001-of-00005.parquet
- data/train-00002-of-00005.parquet
- data/train-00003-of-00005.parquet
- data/train-00004-of-00005.parquet


In [46]:
!pip install -q -U huggingface_hub datasets

from huggingface_hub import notebook_login

notebook_login()

In [47]:
from huggingface_hub import HfApi

api = HfApi()

try:
    info = api.dataset_info("atlasia/Atlaset")
    print("Accès confirmé !")
    print("Dataset :", info.id)
except Exception as e:
    print("Accès non confirmé :", e)

Accès confirmé !
Dataset : atlasia/Atlaset


In [48]:
from huggingface_hub import HfApi

api = HfApi()

try:
    print("Compte connecté :", api.whoami()["name"])

    info = api.dataset_info("atlasia/Atlaset")
    print("Accès aux métadonnées :", "OK")
    print("Dataset :", info.id)

except Exception as e:
    print("Type d'erreur :", type(e).__name__)
    print("Détail :", e)

Compte connecté : ma321as
Accès aux métadonnées : OK
Dataset : atlasia/Atlaset


In [49]:
from datasets import load_dataset

try:
    ds = load_dataset(
        "atlasia/Atlaset",
        split="train",
        streaming=True,
        token=True
    )

    print("Dataset chargé avec succès !")

    first = next(iter(ds))
    print("\nColonnes :", list(first.keys()))
    print("\nPremier exemple :")
    print(first)

except Exception as e:
    print("Erreur :", type(e).__name__)
    print(e)

README.md:   0%|          | 0.00/7.61k [00:00<?, ?B/s]

Dataset chargé avec succès !

Colonnes : ['dataset_source', 'text', 'metadata', 'token_count', 'word_count']

Premier exemple :
{'dataset_source': 'medmac01/moroccan_history_qa', 'text': 'التقريب أكثر من نص مصاريف المخزن كانت كتمشي لبرا باش يخلصو الغرامات ديال الحرب و يشريو السلاح و المعدات العسكرية و السلع المصنعة. من 1902 حتى 1909، العجز التجاري ديال المغرب زاد ب14 مليون فرنك كل عام، و الريال المغربي نقص ب25% من 1896 حتى 1906. في يونيو 1904، من بعد ما فشلات المحاولة ديال فرض ضريبة موحدة، فرنسا عطات قرض للمخزن اللي كان عندو ديون بقيمة 62.5 مليون فرنك، مضمونة بجزء من عائدات الجمارك.', 'metadata': '{\'english\': "More than half of the Makhzen\'s expenditures went abroad to pay war indemnities and buy weapons, military equipment, and manufactured goods.[123] From 1902 to 1909, Morocco\'s trade deficit increased 14 million francs annually, and the Moroccan rial depreciated 25% from 1896 to 1906.[123] In June 1904, after a failed attempt to impose a flat tax, France bailed out the already 

In [50]:
import ast
from itertools import islice
from collections import Counter

# Examiner les 500 premiers enregistrements
records = list(islice(ds, 500))

sources = Counter()
stats = {
    "total": len(records),
    "darija_latn_nonempty": 0,
    "english_nonempty": 0,
    "text_nonempty": 0
}

for record in records:
    sources[record["dataset_source"]] += 1

    # metadata est stockée sous forme de chaîne
    try:
        metadata = ast.literal_eval(record["metadata"])
    except (ValueError, SyntaxError, TypeError):
        metadata = {}

    if metadata.get("darija_Latn", "").strip():
        stats["darija_latn_nonempty"] += 1

    if metadata.get("english", "").strip():
        stats["english_nonempty"] += 1

    if record["text"].strip():
        stats["text_nonempty"] += 1

print("===== DATASET AUDIT =====")
for key, value in stats.items():
    print(f"{key}: {value}")

print("\n===== SOURCES LES PLUS FRÉQUENTES =====")
for source, count in sources.most_common(15):
    print(f"{source}: {count}")

print("\n===== 5 EXEMPLES DE DONNÉES =====")
for i, record in enumerate(records[:5], 1):
    metadata = ast.literal_eval(record["metadata"])
    print(f"\n--- Exemple {i} ---")
    print("Source:", record["dataset_source"])
    print("Texte:", record["text"][:300])
    print("Darija Latin:", metadata.get("darija_Latn", "")[:200])

===== DATASET AUDIT =====
total: 500
darija_latn_nonempty: 0
english_nonempty: 500
text_nonempty: 500

===== SOURCES LES PLUS FRÉQUENTES =====
medmac01/moroccan_history_qa: 500

===== 5 EXEMPLES DE DONNÉES =====

--- Exemple 1 ---
Source: medmac01/moroccan_history_qa
Texte: التقريب أكثر من نص مصاريف المخزن كانت كتمشي لبرا باش يخلصو الغرامات ديال الحرب و يشريو السلاح و المعدات العسكرية و السلع المصنعة. من 1902 حتى 1909، العجز التجاري ديال المغرب زاد ب14 مليون فرنك كل عام، و الريال المغربي نقص ب25% من 1896 حتى 1906. في يونيو 1904، من بعد ما فشلات المحاولة ديال فرض ضريبة 
Darija Latin: 

--- Exemple 2 ---
Source: medmac01/moroccan_history_qa
Texte: ماهي عواقب مصاريف المخزن على الاقتصاد المغربي؟ 
Darija Latin: 

--- Exemple 3 ---
Source: medmac01/moroccan_history_qa
Texte: نفقات المخزن أدت إلى تدهور الاقتصاد المغربي، حيث تم إنفاق جزء كبير من ثروة البلاد على دفع تعويضات الحرب وشراء الأسلحة والمعدات العسكرية والسلع المصنعة. وهذا تسبب في انخفاض كبير في قيمة العملة المغربية، وتراجع الاستثمارات

In [51]:
from collections import Counter
from itertools import islice
import ast

# Recharger le flux depuis le début
ds_audit = load_dataset(
    "atlasia/Atlaset",
    split="train",
    streaming=True,
    token=True
)

records = list(islice(ds_audit, 3000))

sources = Counter()
darija_arabic = 0
darija_latin = 0
english = 0
empty_text = 0

for record in records:
    sources[record["dataset_source"]] += 1

    try:
        metadata = ast.literal_eval(record["metadata"])
    except (ValueError, SyntaxError, TypeError):
        metadata = {}

    if record["text"].strip():
        pass
    else:
        empty_text += 1

    if metadata.get("darija_Latn", "").strip():
        darija_latin += 1

    if metadata.get("english", "").strip():
        english += 1

print("Nombre de lignes analysées :", len(records))
print("Textes vides :", empty_text)
print("Darija Latin renseignée :", darija_latin)
print("English renseigné :", english)

print("\nSources :")
for source, count in sources.most_common():
    print(f"{source}: {count}")

Nombre de lignes analysées : 3000
Textes vides : 0
Darija Latin renseignée : 0
English renseigné : 3000

Sources :
learnmoroccan.com: 1673
medmac01/moroccan_history_qa: 997
loecsen.com: 330


In [52]:
import ast
from collections import defaultdict

selected_sources = [
    "learnmoroccan.com",
    "loecsen.com",
    "medmac01/moroccan_history_qa"
]

examples = defaultdict(list)

for record in records:
    source = record["dataset_source"]

    if source in selected_sources and len(examples[source]) < 5:
        try:
            metadata = ast.literal_eval(record["metadata"])
        except (ValueError, SyntaxError, TypeError):
            metadata = {}

        examples[source].append({
            "text": record["text"],
            "english": metadata.get("english", ""),
            "darija_Latn": metadata.get("darija_Latn", ""),
            "token_count": record["token_count"]
        })

for source in selected_sources:
    print("\n" + "=" * 60)
    print("SOURCE:", source)

    for i, example in enumerate(examples[source], 1):
        print(f"\n--- Exemple {i} ---")
        print("TEXT:", example["text"][:400])
        print("ENGLISH:", example["english"][:300])
        print("DARIJA LATIN:", example["darija_Latn"][:200])


SOURCE: learnmoroccan.com

--- Exemple 1 ---
TEXT: الولد مشى ل مدرسة
ENGLISH: The boy went to school
DARIJA LATIN: 

--- Exemple 2 ---
TEXT: الكتاب ديال ختي
ENGLISH: My sister's book
DARIJA LATIN: 

--- Exemple 3 ---
TEXT: أنا غادي ل الدار
ENGLISH: I'm going to the house
DARIJA LATIN: 

--- Exemple 4 ---
TEXT: الكتاب كاين في صاك.
ENGLISH: The book is in the bag.
DARIJA LATIN: 

--- Exemple 5 ---
TEXT: هداك الراجل كبير
ENGLISH: That man is tall
DARIJA LATIN: 

SOURCE: loecsen.com

--- Exemple 1 ---
TEXT: لبار
ENGLISH: The bar
DARIJA LATIN: 

--- Exemple 2 ---
TEXT: تشرب شي حاجة؟ 
ENGLISH: Would you like to have a drink?
DARIJA LATIN: 

--- Exemple 3 ---
TEXT: تشربي شي حاجة؟ 
ENGLISH: Would you like to have a drink?
DARIJA LATIN: 

--- Exemple 4 ---
TEXT: كنشرب
ENGLISH: I'm drinking
DARIJA LATIN: 

--- Exemple 5 ---
TEXT: لكاس
ENGLISH: a glass
DARIJA LATIN: 

SOURCE: medmac01/moroccan_history_qa

--- Exemple 1 ---
TEXT: التقريب أكثر من نص مصاريف المخزن كانت كتمشي لبرا باش يخلصو الغرامات

In [53]:
import json
from pathlib import Path

# Vérifier que le dossier Bronze existe
BRONZE_DIR = Path("/content/Moroccan_Darija_LLM/data/bronze")
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

# Fichier contenant les 3000 lignes brutes
output_file = BRONZE_DIR / "atlaset_train_sample_3000.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Sauvegarde terminée !")
print("Fichier :", output_file)
print("Nombre de lignes :", len(records))
print("Taille :", round(output_file.stat().st_size / 1024, 2), "KB")

Sauvegarde terminée !
Fichier : /content/Moroccan_Darija_LLM/data/bronze/atlaset_train_sample_3000.jsonl
Nombre de lignes : 3000
Taille : 1641.21 KB


In [54]:
import re
import pandas as pd
from collections import Counter

def detect_script(text):
    """Détecte les principaux alphabets utilisés."""
    if not isinstance(text, str) or not text.strip():
        return "empty"

    has_arabic = bool(re.search(r"[\u0600-\u06FF]", text))
    has_latin = bool(re.search(r"[A-Za-z]", text))

    if has_arabic and has_latin:
        return "mixed_arabic_latin"
    elif has_arabic:
        return "arabic"
    elif has_latin:
        return "latin"
    else:
        return "other"


def detect_text_type(text):
    """Détecte la structure élémentaire du texte."""
    text = text.strip()

    if not text:
        return "empty"

    if "؟" in text or "?" in text:
        return "question"

    if len(text.split()) <= 4:
        return "short_expression"

    return "sentence"


# Construire le DataFrame depuis nos données Bronze
df = pd.DataFrame(records)

df["script_type"] = df["text"].apply(detect_script)
df["text_type"] = df["text"].apply(detect_text_type)
df["char_count"] = df["text"].str.len()
df["word_count_calc"] = df["text"].str.split().str.len()

print("===== SCRIPT DISTRIBUTION =====")
print(df["script_type"].value_counts())

print("\n===== TEXT TYPE DISTRIBUTION =====")
print(df["text_type"].value_counts())

print("\n===== STATISTICS =====")
print(df[["char_count", "word_count_calc", "token_count"]].describe())

===== SCRIPT DISTRIBUTION =====
script_type
arabic                2993
mixed_arabic_latin       7
Name: count, dtype: int64

===== TEXT TYPE DISTRIBUTION =====
text_type
short_expression    1354
sentence            1140
question             506
Name: count, dtype: int64

===== STATISTICS =====
        char_count  word_count_calc  token_count
count  3000.000000      3000.000000  3000.000000
mean     84.687333        14.435667    21.598000
std     167.657287        27.696839    41.636037
min       1.000000         1.000000     1.000000
25%      10.000000         2.000000     3.000000
50%      23.000000         5.000000     7.000000
75%      58.000000        10.000000    15.000000
max    1742.000000       288.000000   430.000000


In [55]:
print("===== SOURCE × SCRIPT =====")

print(
    pd.crosstab(
        df["dataset_source"],
        df["script_type"],
        margins=True
    )
)

print("\n===== SOURCE × TEXT TYPE =====")

print(
    pd.crosstab(
        df["dataset_source"],
        df["text_type"],
        margins=True
    )
)

===== SOURCE × SCRIPT =====
script_type                   arabic  mixed_arabic_latin   All
dataset_source                                                
learnmoroccan.com               1673                   0  1673
loecsen.com                      330                   0   330
medmac01/moroccan_history_qa     990                   7   997
All                             2993                   7  3000

===== SOURCE × TEXT TYPE =====
text_type                     question  sentence  short_expression   All
dataset_source                                                          
learnmoroccan.com                  105       491              1077  1673
loecsen.com                         74         7               249   330
medmac01/moroccan_history_qa       327       642                28   997
All                                506      1140              1354  3000


In [56]:
# 1. Examiner les 7 exemples avec écriture arabe + latine
print("===== EXEMPLES ARABE + LATIN =====")

mixed_examples = df[df["script_type"] == "mixed_arabic_latin"]

if len(mixed_examples) > 0:
    for _, row in mixed_examples.iterrows():
        print("\nSource :", row["dataset_source"])
        print("Texte  :", row["text"])
else:
    print("Aucun exemple mixte trouvé.")


# 2. Examiner 5 exemples aléatoires par source
print("\n\n===== EXEMPLES PAR SOURCE =====")

for source in df["dataset_source"].dropna().unique():
    print(f"\n\n### SOURCE : {source}")

    sample = (
        df[df["dataset_source"] == source]
        .sample(n=min(5, len(df[df["dataset_source"] == source])),
                random_state=42)
    )

    for _, row in sample.iterrows():
        print("\nTexte :", row["text"])
        print("Nombre de mots :", row["word_count_calc"])

        # Afficher les informations disponibles dans les métadonnées
        metadata = row.get("metadata")

        if isinstance(metadata, dict):
            print("Métadonnées :", metadata)
        elif isinstance(metadata, str):
            print("Métadonnées :", metadata[:500])

===== EXEMPLES ARABE + LATIN =====

Source : medmac01/moroccan_history_qa
Texte  : شنو كان النتيجة ديال منتدى Connect لتطوير الخطوط الجوية في طنجة في 2023؟ 

Source : medmac01/moroccan_history_qa
Texte  : كاين بزاف ديال الجامعات داخل و برا المدينة. جامعات بحال المعهد العالي الدولي للسياحة (ISIT)، لي كيعطي الديبلومات، كيقدم دروس من تدبير الأعمال حتى لتسيير الفنادق. المعهد هو واحد من أرقى مدارس السياحة فالبلاد. كاين جامعات خرى بحال المدرسة الوطنية للتجارة والتسيير (ENCG-T) لي هي من أكبر مدارس الأعمال فالبلاد وحتى المدرسة الوطنية للعلوم التطبيقية (ENSA-T)، لي هي مدرسة مهندسين زايدة فالعلوم التطبيقية. الجامعة المعروفة بعبد المالك السعدي فيها بزاف ديال الكليات؛ كلية الحقوق والإقتصاد والعلوم الإجتماعية (FSJEST) وكلية العلوم التقنية (FST). و المعهد لي كيشد أكثر ناس هو معهد ISTA التابع ل OFPPT.

Source : medmac01/moroccan_history_qa
Texte  : شنو نوع د المتحف ديال Fondation Lorin فطنجة؟ 

Source : medmac01/moroccan_history_qa
Texte  : الرباط و تمارة و سلا كيشكلو تجمع حضري ديال أكثر من 1.8 مليون

In [57]:
import ast
import pandas as pd
from collections import Counter

# Convertir les métadonnées en dictionnaires Python
def parse_metadata(value):
    if isinstance(value, dict):
        return value
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError, TypeError):
        return {}

df["metadata_parsed"] = df["metadata"].apply(parse_metadata)

# Extraire les champs intéressants
metadata_df = pd.json_normalize(df["metadata_parsed"])

print("===== METADATA FIELDS =====")
print(metadata_df.columns.tolist())

print("\n===== ROLE DISTRIBUTION =====")
print(metadata_df["role"].value_counts(dropna=False))

print("\n===== TOPICS =====")
print(metadata_df["topic"].value_counts(dropna=False).head(15))

print("\n===== SUBTOPICS =====")
print(metadata_df["subtopic"].value_counts(dropna=False).head(15))

print("\n===== NON-EMPTY DARija_LATN =====")
print(
    metadata_df["darija_Latn"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)

===== METADATA FIELDS =====
['english', 'darija_Latn', 'dataset_source', 'id', 'role', 'darija_tokens', 'subtopic', 'topic', 'annotator_dialect']

===== ROLE DISTRIBUTION =====
role
None    3000
Name: count, dtype: int64

===== TOPICS =====
topic
    3000
Name: count, dtype: int64

===== SUBTOPICS =====
subtopic
    3000
Name: count, dtype: int64

===== NON-EMPTY DARija_LATN =====
0


In [58]:
import pandas as pd

# Nettoyage technique temporaire pour l'audit
df["text_check"] = (
    df["text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["english_check"] = (
    df["metadata_parsed"]
    .apply(lambda x: str(x.get("english", "") or "").strip())
)

# 1. Valeurs manquantes
print("===== MISSING VALUES =====")
print("Empty Darija:", (df["text_check"] == "").sum())
print("Empty English:", (df["english_check"] == "").sum())

# 2. Doublons exacts
print("\n===== DUPLICATES =====")
print("Duplicate Darija:", df["text_check"].duplicated().sum())
print("Duplicate English:", df["english_check"].duplicated().sum())

# 3. Textes très courts
print("\n===== TEXT LENGTH =====")
print("Darija <= 2 words:", (df["text_check"].str.split().str.len() <= 2).sum())
print("Darija <= 4 words:", (df["text_check"].str.split().str.len() <= 4).sum())

# 4. Exemples de doublons Darija
print("\n===== DUPLICATE EXAMPLES =====")
duplicates = df[df["text_check"].duplicated(keep=False)]

print(
    duplicates[
        ["dataset_source", "text_check", "english_check"]
    ].head(15).to_string(index=False)
)

===== MISSING VALUES =====
Empty Darija: 0
Empty English: 0

===== DUPLICATES =====
Duplicate Darija: 1
Duplicate English: 28

===== TEXT LENGTH =====
Darija <= 2 words: 795
Darija <= 4 words: 1465

===== DUPLICATE EXAMPLES =====
dataset_source text_check                english_check
   loecsen.com    فهمتي ؟ therefore do you understand?
   loecsen.com    فهمتي ؟      thus do you understand?


In [59]:
import re
import unicodedata

def normalize_for_dedup(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    # Supprimer les espaces avant la ponctuation
    text = re.sub(r"\s+([؟?!.,،؛:])", r"\1", text)

    return text

df["text_normalized"] = df["text_check"].apply(normalize_for_dedup)

print("===== NORMALIZED DUPLICATES =====")

duplicate_mask = df["text_normalized"].duplicated(keep=False)

print(
    "Number of rows involved:",
    duplicate_mask.sum()
)

print(
    "Number of duplicate rows after first occurrence:",
    df["text_normalized"].duplicated().sum()
)

print("\n===== DUPLICATE GROUPS =====")

duplicate_examples = df[duplicate_mask].sort_values("text_normalized")

print(
    duplicate_examples[
        ["dataset_source", "text_check", "text_normalized", "english_check"]
    ].to_string(index=False)
)

===== NORMALIZED DUPLICATES =====
Number of rows involved: 2
Number of duplicate rows after first occurrence: 1

===== DUPLICATE GROUPS =====
dataset_source text_check text_normalized                english_check
   loecsen.com    فهمتي ؟          فهمتي؟ therefore do you understand?
   loecsen.com    فهمتي ؟          فهمتي؟      thus do you understand?


In [60]:
import re
import unicodedata

def normalize_for_dedup(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)

    # Supprimer les espaces avant la ponctuation
    text = re.sub(r"\s+([؟?!.,،؛:])", r"\1", text)

    return text

df["text_normalized"] = df["text_check"].apply(normalize_for_dedup)

print("===== NORMALIZED DUPLICATES =====")

duplicate_mask = df["text_normalized"].duplicated(keep=False)

print(
    "Number of rows involved:",
    duplicate_mask.sum()
)

print(
    "Number of duplicate rows after first occurrence:",
    df["text_normalized"].duplicated().sum()
)

print("\n===== DUPLICATE GROUPS =====")

duplicate_examples = df[duplicate_mask].sort_values("text_normalized")

print(
    duplicate_examples[
        ["dataset_source", "text_check", "text_normalized", "english_check"]
    ].to_string(index=False)
)

===== NORMALIZED DUPLICATES =====
Number of rows involved: 2
Number of duplicate rows after first occurrence: 1

===== DUPLICATE GROUPS =====
dataset_source text_check text_normalized                english_check
   loecsen.com    فهمتي ؟          فهمتي؟ therefore do you understand?
   loecsen.com    فهمتي ؟          فهمتي؟      thus do you understand?


In [61]:
import json
from pathlib import Path

# Dossiers Silver
SILVER_DIR = Path("/content/Moroccan_Darija_LLM/data/silver")
SILVER_DIR.mkdir(parents=True, exist_ok=True)

# Préparer les données nettoyées
silver_df = df.copy()

silver_df["text"] = silver_df["text_normalized"]
silver_df["english"] = silver_df["english_check"]

# Déduplication par source et texte Darija normalisé
silver_df = silver_df.drop_duplicates(
    subset=["dataset_source", "text_normalized"],
    keep="first"
)

# Séparer les sources
source_files = {
    "learnmoroccan.com": "silver_learnmoroccan.jsonl",
    "loecsen.com": "silver_loecsen.jsonl",
    "medmac01/moroccan_history_qa": "silver_history.jsonl"
}

# Exporter chaque source en JSONL
for source, filename in source_files.items():

    source_df = silver_df[
        silver_df["dataset_source"] == source
    ]

    output_path = SILVER_DIR / filename

    with open(output_path, "w", encoding="utf-8") as f:
        for _, row in source_df.iterrows():

            record = {
                "text": row["text"],
                "english": row["english"],
                "dataset_source": row["dataset_source"],
                "script_type": row["script_type"],
                "text_type": row["text_type"],
                "token_count": row["token_count"],
                "metadata": row["metadata_parsed"]
            }

            f.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    print(f"{filename}: {len(source_df)} lignes")

print("\nTotal Silver:", len(silver_df))
print("Dossier:", SILVER_DIR)

silver_learnmoroccan.jsonl: 1673 lignes
silver_loecsen.jsonl: 329 lignes
silver_history.jsonl: 997 lignes

Total Silver: 2999
Dossier: /content/Moroccan_Darija_LLM/data/silver


In [62]:
import json
from pathlib import Path
from collections import Counter

SILVER_DIR = Path("/content/Moroccan_Darija_LLM/data/silver")

files = [
    "silver_learnmoroccan.jsonl",
    "silver_loecsen.jsonl",
    "silver_history.jsonl"
]

total_records = 0

for filename in files:
    path = SILVER_DIR / filename

    records_check = []
    errors = []

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            try:
                record = json.loads(line)
                records_check.append(record)

                # Vérifier les champs obligatoires
                required = ["text", "english", "dataset_source"]

                for field in required:
                    if not record.get(field):
                        errors.append(
                            f"Ligne {line_number}: champ {field} vide"
                        )

            except json.JSONDecodeError as e:
                errors.append(f"Ligne {line_number}: JSON invalide ({e})")

    total_records += len(records_check)

    print(f"\n===== {filename} =====")
    print("Nombre de lignes :", len(records_check))
    print("Erreurs :", len(errors))
    print("Exemple :", records_check[0] if records_check else "Fichier vide")

    if errors:
        print("Détails :", errors[:5])

print("\n===== BILAN GLOBAL =====")
print("Total des enregistrements :", total_records)


===== silver_learnmoroccan.jsonl =====
Nombre de lignes : 1673
Erreurs : 0
Exemple : {'text': 'الولد مشى ل مدرسة', 'english': 'The boy went to school', 'dataset_source': 'learnmoroccan.com', 'script_type': 'arabic', 'text_type': 'short_expression', 'token_count': 5, 'metadata': {'english': 'The boy went to school', 'darija_Latn': '', 'dataset_source': 'learnmoroccan.com', 'id': 'None', 'role': 'None', 'darija_tokens': '10', 'subtopic': '', 'topic': '', 'annotator_dialect': ''}}

===== silver_loecsen.jsonl =====
Nombre de lignes : 329
Erreurs : 0
Exemple : {'text': 'لبار', 'english': 'The bar', 'dataset_source': 'loecsen.com', 'script_type': 'arabic', 'text_type': 'short_expression', 'token_count': 2, 'metadata': {'english': 'The bar', 'darija_Latn': '', 'dataset_source': 'loecsen.com', 'id': 'None', 'role': 'None', 'darija_tokens': '4', 'subtopic': '', 'topic': '', 'annotator_dialect': ''}}

===== silver_history.jsonl =====
Nombre de lignes : 997
Erreurs : 0
Exemple : {'text': 'التقري

In [63]:
import pandas as pd

sources = [
    "learnmoroccan.com",
    "loecsen.com",
    "medmac01/moroccan_history_qa"
]

for source in sources:
    sample = (
        df[df["dataset_source"] == source]
        .sample(n=min(10, len(df[df["dataset_source"] == source])),
                random_state=42)
    )

    print(f"\n{'=' * 25}")
    print(f"SOURCE : {source}")
    print("=" * 25)

    for i, (_, row) in enumerate(sample.iterrows(), start=1):
        print(f"\n{i}. Darija : {row['text_check']}")
        print(f"   English : {row['english_check']}")


SOURCE : learnmoroccan.com

1. Darija : هاد الولد مسكين، ما بغاوش يقبلو ف المدرسة
   English : This poor boy, they didn't want to accept him in school

2. Darija : حط الما فالتلاجة
   English : Put the water in the fridge

3. Darija : خاصك تنوض دابا
   English : You need to get up now

4. Darija : شريت واحد لوحة زوينة ل دار.
   English : I bought a beautiful painting for the house.

5. Darija : وقع
   English : Occur

6. Darija : هاد الكتاب مهم بزاف لقرايتك.
   English : This book is very important for your studies.

7. Darija : ما بقاش العنف بزاف
   English : There isn't much violence anymore

8. Darija : هاد القهوة مزيانة بزاف
   English : This coffee is very good

9. Darija : أنا عضو في نادي كرة القدم
   English : I am a member of a football club

10. Darija : حوايجي كاينين ف الداخل
   English : My things are inside

SOURCE : loecsen.com

1. Darija : ممكن تزيد مربعات ديال لكلاصون عافاك؟
   English : Can you add some ice cubes, please?

2. Darija : كنلعب التنيس
   English : I play t

In [64]:
import json
from pathlib import Path

# 1. Sélectionner les deux sources adaptées à la Darija quotidienne
daily_candidates = silver_df[
    silver_df["dataset_source"].isin([
        "learnmoroccan.com",
        "loecsen.com"
    ])
].copy()

# 2. Ajouter des colonnes pour notre contrôle qualité
daily_candidates["quality_status"] = "pending"
daily_candidates["linguistic_review"] = ""
daily_candidates["dataset_category"] = "daily_darija"

# 3. Préparer le dossier de sortie
output_dir = Path(
    "/content/Moroccan_Darija_LLM/data/silver"
)
output_dir.mkdir(parents=True, exist_ok=True)

# 4. Enregistrer en JSONL
output_file = output_dir / "silver_daily_candidates.jsonl"

with open(output_file, "w", encoding="utf-8") as f:
    for record in daily_candidates.to_dict(orient="records"):
        f.write(
            json.dumps(record, ensure_ascii=False, default=str) + "\n"
        )

# 5. Vérifier le résultat
print("Nombre de candidats :", len(daily_candidates))
print("\nRépartition par source :")
print(daily_candidates["dataset_source"].value_counts())

print("\nStatut qualité :")
print(daily_candidates["quality_status"].value_counts())

print("\nFichier créé :", output_file)


Nombre de candidats : 2002

Répartition par source :
dataset_source
learnmoroccan.com    1673
loecsen.com           329
Name: count, dtype: int64

Statut qualité :
quality_status
pending    2002
Name: count, dtype: int64

Fichier créé : /content/Moroccan_Darija_LLM/data/silver/silver_daily_candidates.jsonl


In [65]:
import pandas as pd
from pathlib import Path

# Copier les candidats sans modifier les données originales
review_df = daily_candidates.copy()

# Ajouter les colonnes de validation
review_df["quality_status"] = "pending"
review_df["corrected_text"] = ""
review_df["corrected_english"] = ""
review_df["review_comment"] = ""

# Ajouter un identifiant unique
review_df.insert(
    0,
    "review_id",
    [f"DRJ_{i:04d}" for i in range(1, len(review_df) + 1)]
)

# Organiser les colonnes
review_df = review_df[
    [
        "review_id",
        "dataset_source",
        "text",
        "english",
        "text_type",
        "quality_status",
        "corrected_text",
        "corrected_english",
        "review_comment"
    ]
]

# Créer le dossier
review_dir = Path(
    "/content/Moroccan_Darija_LLM/data/silver"
)
review_dir.mkdir(parents=True, exist_ok=True)

# Exporter le fichier Excel
excel_path = review_dir / "darija_manual_review.xlsx"

review_df.to_excel(
    excel_path,
    index=False
)

# Exporter aussi en CSV
csv_path = review_dir / "darija_manual_review.csv"

review_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

print("Nombre total à vérifier :", len(review_df))
print("Excel :", excel_path)
print("CSV :", csv_path)

Nombre total à vérifier : 2002
Excel : /content/Moroccan_Darija_LLM/data/silver/darija_manual_review.xlsx
CSV : /content/Moroccan_Darija_LLM/data/silver/darija_manual_review.csv


In [66]:
import json
from pathlib import Path

# 1. Marquer les 2 002 exemples comme validés
daily_candidates["quality_status"] = "accepted"
daily_candidates["linguistic_review"] = "validated_by_user"

# 2. Préparer le dossier Gold
gold_dir = Path(
    "/content/Moroccan_Darija_LLM/data/gold"
)
gold_dir.mkdir(parents=True, exist_ok=True)

# 3. Conserver uniquement les données validées
gold_df = daily_candidates[
    daily_candidates["quality_status"] == "accepted"
].copy()

# 4. Sauvegarder le dataset Gold en JSONL
gold_path = gold_dir / "darija_daily_validated.jsonl"

gold_df.to_json(
    gold_path,
    orient="records",
    lines=True,
    force_ascii=False
)

# 5. Vérifier l'intégrité
with open(gold_path, "r", encoding="utf-8") as f:
    gold_records = [json.loads(line) for line in f]

print("Nombre d'exemples Gold :", len(gold_records))
print("Sources :")
print(gold_df["dataset_source"].value_counts())
print("Statuts :")
print(gold_df["quality_status"].value_counts())
print("Fichier :", gold_path)

Nombre d'exemples Gold : 2002
Sources :
dataset_source
learnmoroccan.com    1673
loecsen.com           329
Name: count, dtype: int64
Statuts :
quality_status
accepted    2002
Name: count, dtype: int64
Fichier : /content/Moroccan_Darija_LLM/data/gold/darija_daily_validated.jsonl


In [67]:
import pandas as pd
import re

# Charger le dataset Gold
gold_df = pd.read_json(
    "/content/Moroccan_Darija_LLM/data/gold/darija_daily_validated.jsonl",
    lines=True
)

# Détection du type d'écriture
def detect_script(text):
    text = str(text)

    has_arabic = bool(re.search(r"[\u0600-\u06FF]", text))
    has_latin = bool(re.search(r"[a-zA-Z]", text))

    if has_arabic and has_latin:
        return "mixed_arabic_latin"
    elif has_arabic:
        return "arabic"
    elif has_latin:
        return "latin"
    else:
        return "other"

gold_df["script_type"] = gold_df["text"].apply(detect_script)

# Longueur des textes
gold_df["word_count"] = gold_df["text"].astype(str).apply(
    lambda x: len(x.split())
)

# Analyse de la couverture
print("===== GOLD DATASET PROFILE =====")
print("Total examples:", len(gold_df))

print("\n===== SCRIPT DISTRIBUTION =====")
print(gold_df["script_type"].value_counts())

print("\n===== TEXT TYPE DISTRIBUTION =====")
print(gold_df["text_type"].value_counts())

print("\n===== WORD COUNT STATISTICS =====")
print(gold_df["word_count"].describe())

print("\n===== TRANSLATION COVERAGE =====")
print(
    "English translations available:",
    gold_df["english"].notna().sum()
)
print(
    "Empty English translations:",
    gold_df["english"].isna().sum()
)

print("\n===== SAMPLE LATIN / MIXED TEXTS =====")
display(
    gold_df[
        gold_df["script_type"].isin(
            ["latin", "mixed_arabic_latin"]
        )
    ][["text", "english"]].head(15)
)

===== GOLD DATASET PROFILE =====
Total examples: 2002

===== SCRIPT DISTRIBUTION =====
script_type
arabic    2002
Name: count, dtype: int64

===== TEXT TYPE DISTRIBUTION =====
text_type
short_expression    1326
sentence             498
question             178
Name: count, dtype: int64

===== WORD COUNT STATISTICS =====
count    2002.000000
mean        3.246753
std         1.958605
min         1.000000
25%         1.000000
50%         3.000000
75%         5.000000
max        10.000000
Name: word_count, dtype: float64

===== TRANSLATION COVERAGE =====
English translations available: 2002
Empty English translations: 0

===== SAMPLE LATIN / MIXED TEXTS =====


,text,english


In [68]:
import pandas as pd
from pathlib import Path

# URL du fichier officiel DODa
doda_url = (
    "https://raw.githubusercontent.com/"
    "darija-open-dataset/dataset/main/"
    "sentences/sentences.csv"
)

# Télécharger et lire le CSV
doda_df = pd.read_csv(
    doda_url,
    encoding="utf-8",
    low_memory=False
)

# Sauvegarder une copie Bronze
bronze_dir = Path(
    "/content/Moroccan_Darija_LLM/data/bronze"
)
bronze_dir.mkdir(parents=True, exist_ok=True)

bronze_path = bronze_dir / "doda_sentences_raw.csv"
doda_df.to_csv(bronze_path, index=False, encoding="utf-8")

# Première inspection
print("===== DODA DATASET =====")
print("Nombre de lignes :", len(doda_df))
print("Nombre de colonnes :", len(doda_df.columns))

print("\n===== COLONNES =====")
print(doda_df.columns.tolist())

print("\n===== TYPES DE DONNÉES =====")
print(doda_df.dtypes)

print("\n===== 10 PREMIÈRES LIGNES =====")
display(doda_df.head(10))

print("\nFichier Bronze sauvegardé :", bronze_path)

===== DODA DATASET =====
Nombre de lignes : 48839
Nombre de colonnes : 3

===== COLONNES =====
['darija', 'eng', 'darija_ar']

===== TYPES DE DONNÉES =====
darija       object
eng          object
darija_ar    object
dtype: object

===== 10 PREMIÈRES LIGNES =====


,darija,eng,darija_ar
0,"homa mkhbbyin chi haja, ana mti99en!","They're hiding something, I'm sure!","هوما مخبّين شي حاجة, أنا متيقّن!"
1,bayna homa tay7awlo ib9aw mbrrdin.,It's obvious they're trying to keep their cool.,باينا هوما تايحاولو إبقاو مبرّدين.
2,loTilat mabaynach fihom mori7in bzzaf.,the hotels don't seem very comfortable.,لوطيلات مابايناش فيهوم مريحين بزّاف.
3,ghaliban ghayjrriw 3lih mn lkhdma!,he is probably about to be laid off by head of...,غالبا غايجرّيو عليه من لخدمة!
4,Tab3an rah mkta2eb!,of course he's depressive!,طبعا راه مكتاءب!
5,tayballia ghanmchi!,I think I'm going to leave!,تايبالّيا غانمشي!
6,ara lia dak sac.,hand me that bag.,أرا ليا داك ساك.
7,ghanmreD!,I'm going to be sick!,غانمرض!
8,knt dima 3arf annaha bghatna nmoutou,I always knew she wanted us dead,كنت ديما عارف أنّها بغاتنا نموتو
9,bghit n3ref ch7al b9a lia tlwe9t bach n9ra,I'd like to know how much time I have left to ...,بغيت نعرف شحال بقا ليا تلوقت باش نقرا



Fichier Bronze sauvegardé : /content/Moroccan_Darija_LLM/data/bronze/doda_sentences_raw.csv


In [69]:
import pandas as pd
import re

# Travailler sur une copie
doda_profile = doda_df.copy()

# 1. Nettoyage technique temporaire pour l'analyse
for col in ["darija", "darija_ar", "eng"]:
    doda_profile[col] = (
        doda_profile[col]
        .astype("string")
        .str.strip()
    )

# 2. Valeurs manquantes et vides
print("===== MISSING VALUES =====")

for col in ["darija", "darija_ar", "eng"]:
    missing = (
        doda_profile[col].isna()
        | doda_profile[col].eq("")
    ).sum()

    print(f"{col}: {missing}")

# 3. Doublons
print("\n===== DUPLICATES =====")

print(
    "Doublons Darija Latin:",
    doda_profile["darija"].duplicated().sum()
)

print(
    "Doublons Darija Arabic:",
    doda_profile["darija_ar"].duplicated().sum()
)

print(
    "Doublons sur les trois colonnes:",
    doda_profile.duplicated(
        subset=["darija", "eng", "darija_ar"]
    ).sum()
)

# 4. Longueur des textes
for col in ["darija", "darija_ar", "eng"]:
    doda_profile[f"{col}_words"] = (
        doda_profile[col]
        .fillna("")
        .apply(lambda x: len(x.split()))
    )

print("\n===== TEXT LENGTH =====")

display(
    doda_profile[
        ["darija_words", "darija_ar_words", "eng_words"]
    ].describe()
)

# 5. Vérification des scripts
def detect_script(text):
    text = str(text)

    arabic = bool(re.search(r"[\u0600-\u06FF]", text))
    latin = bool(re.search(r"[a-zA-Z]", text))

    if arabic and latin:
        return "mixed"
    elif arabic:
        return "arabic"
    elif latin:
        return "latin"
    return "other"

doda_profile["darija_script"] = (
    doda_profile["darija"].apply(detect_script)
)

print("\n===== SCRIPT DISTRIBUTION =====")
print(doda_profile["darija_script"].value_counts())

# 6. Vérifier les lignes complètes
complete = doda_profile.dropna(
    subset=["darija", "darija_ar", "eng"]
)

complete = complete[
    (complete["darija"] != "")
    & (complete["darija_ar"] != "")
    & (complete["eng"] != "")
]

print("\n===== COMPLETE RECORDS =====")
print("Lignes complètes :", len(complete))
print("Lignes incomplètes :", len(doda_profile) - len(complete))

# 7. Afficher 15 exemples aléatoires
print("\n===== RANDOM SAMPLE =====")

display(
    complete[
        ["darija", "darija_ar", "eng"]
    ].sample(
        n=min(15, len(complete)),
        random_state=42
    )
)


===== MISSING VALUES =====
darija: 0
darija_ar: 0
eng: 21

===== DUPLICATES =====
Doublons Darija Latin: 335
Doublons Darija Arabic: 981
Doublons sur les trois colonnes: 77

===== TEXT LENGTH =====


,darija_words,darija_ar_words,eng_words
count,48839.000000,48839.000000,48839.000000
mean,4.528676,4.474457,5.408301
std,2.343324,2.311796,2.533906
min,1.000000,1.000000,0.000000
25%,3.000000,3.000000,4.000000
50%,4.000000,4.000000,5.000000
75%,6.000000,6.000000,7.000000
max,29.000000,29.000000,38.000000



===== SCRIPT DISTRIBUTION =====
darija_script
latin    48839
Name: count, dtype: int64

===== COMPLETE RECORDS =====
Lignes complètes : 48818
Lignes incomplètes : 21

===== RANDOM SAMPLE =====


,darija,darija_ar,eng
3030,aykhasak tji tzorni 9riban,عيخصك تجي تزورني قريباً,You must come visit me soon
3610,"fnfs lw9t, imkan lina ndwiw 3la lkhoTat dyalna...",فنفس الوقت، إمكان لينا ندويو على الخطط ديالنا ...,"In the meantime, we could talk about our futur..."
5676,3ndak 15 lyom 9bal nhar l3ard t9dimi bach tSaw...,عندك 15 يوم قبل نهار العرض تقديم باش تصوب شغلك,You've got 15 days before the presentation dat...
21776,kantsenna fik t dowwer bhadik lblassa chwak ba...,كانتسنّا فيك ت دور باديك لبلاصة شوك باش مايدخل...,I'm waiting for you to encercle this place
14754,9ari 7aasim.,قاري حسيم.,study hassim
9311,ghber,غبر,Get lost
26387,moul l9ehwa jra 3liya mora maheresst chi kissa...,مول القهوة جرا عليا مورا ماهرسّت شي كيسان وست ...,coffee has taken me out
1406,"ayeh, sir 3ellah, njerbo!","أيه, سير عالله, نجربو!","Yes, go ahead, we can try!"
40799,ghandi nbghik trj3 3ndi ta gheda,غاندي نبغيك ترجع عندي تا غدا,i want to come back tomorrow
15136,rah ja godammek hadak l7anout,راه جا ڭدّامّك هاداك لحانوت,this place is in front of shop


In [70]:
# 1. Doublons exacts sur les trois colonnes
exact_duplicates = doda_profile[
    doda_profile.duplicated(
        subset=["darija", "darija_ar", "eng"],
        keep=False
    )
].sort_values(["darija", "eng"])

print("===== EXACT DUPLICATES =====")
print("Lignes concernées :", len(exact_duplicates))

# 2. Même Darija latine, plusieurs traductions anglaises
latin_groups = (
    doda_profile
    .groupby("darija")["eng"]
    .nunique()
)

multiple_english = latin_groups[latin_groups > 1]

print("\n===== SAME LATIN, DIFFERENT ENGLISH =====")
print("Expressions concernées :", len(multiple_english))

# 3. Même Darija arabe, plusieurs traductions anglaises
arabic_groups = (
    doda_profile
    .groupby("darija_ar")["eng"]
    .nunique()
)

multiple_arabic = arabic_groups[arabic_groups > 1]

print("\n===== SAME ARABIC, DIFFERENT ENGLISH =====")
print("Expressions concernées :", len(multiple_arabic))

# 4. Afficher quelques exemples à examiner
print("\n===== EXAMPLES TO REVIEW =====")

if len(multiple_english) > 0:
    examples = (
        doda_profile[
            doda_profile["darija"].isin(
                multiple_english.index[:10]
            )
        ][["darija", "darija_ar", "eng"]]
        .sort_values("darija")
    )

    display(examples.head(30))

===== EXACT DUPLICATES =====
Lignes concernées : 140

===== SAME LATIN, DIFFERENT ENGLISH =====
Expressions concernées : 217

===== SAME ARABIC, DIFFERENT ENGLISH =====
Expressions concernées : 441

===== EXAMPLES TO REVIEW =====


,darija,darija_ar,eng
4818,3afak,عفاك,Please
10274,3afak,عفاك,please
11076,3awd 3afak,عاود عفاك,repeat please
6730,3awd 3afak,عاود عفاك,Repeat please
6731,3awdi 3afak,عاودي عفاك,Repeat please
11077,3awdi 3afak,عاودي عفاك,repeat please
4110,3awnni 3afak,عاوني عفاك,"Help me, please!"
7641,3awnni 3afak,عاوني عفاك,help me please
4143,3azizi,عزيزي,my dear
6245,3azizi,عزيزي,Oh dear!


In [71]:
import pandas as pd
from pathlib import Path

# 1. Charger une copie des données Bronze
doda_silver = doda_df.copy()

# 2. Conserver l'identifiant de la ligne source
doda_silver.insert(
    0,
    "source_row_id",
    range(1, len(doda_silver) + 1)
)

# 3. Nettoyer uniquement les espaces autour des textes
for col in ["darija", "darija_ar", "eng"]:
    doda_silver[col] = (
        doda_silver[col]
        .astype("string")
        .str.strip()
    )

# 4. Identifier les lignes incomplètes
missing_mask = (
    doda_silver["darija"].isna()
    | doda_silver["darija"].eq("")
    | doda_silver["darija_ar"].isna()
    | doda_silver["darija_ar"].eq("")
    | doda_silver["eng"].isna()
    | doda_silver["eng"].eq("")
)

doda_quarantine = doda_silver[missing_mask].copy()

# 5. Garder uniquement les lignes complètes
doda_silver = doda_silver[~missing_mask].copy()

# 6. Supprimer les doublons strictement identiques
# On compare les 3 textes, pas uniquement la Darija.
before_dedup = len(doda_silver)

doda_silver = doda_silver.drop_duplicates(
    subset=["darija", "darija_ar", "eng"],
    keep="first"
).copy()

removed_duplicates = before_dedup - len(doda_silver)

# 7. Ajouter les métadonnées de provenance
doda_silver["dataset_source"] = "DODa"
doda_silver["quality_status"] = "pending"

# 8. Créer les dossiers
silver_dir = Path(
    "/content/Moroccan_Darija_LLM/data/silver"
)
silver_dir.mkdir(parents=True, exist_ok=True)

# 9. Enregistrer le Silver
silver_path = silver_dir / "doda_silver.jsonl"

doda_silver.to_json(
    silver_path,
    orient="records",
    lines=True,
    force_ascii=False
)

# 10. Mettre les lignes incomplètes en quarantaine
quarantine_path = silver_dir / "doda_quarantine.jsonl"

doda_quarantine.to_json(
    quarantine_path,
    orient="records",
    lines=True,
    force_ascii=False
)

# 11. Rapport final
print("===== DODA SILVER REPORT =====")
print("Bronze original :", len(doda_df))
print("Lignes incomplètes :", len(doda_quarantine))
print("Doublons exacts supprimés :", removed_duplicates)
print("Lignes Silver :", len(doda_silver))

print("\n===== FILES =====")
print("Silver :", silver_path)
print("Quarantine :", quarantine_path)

print("\n===== QUALITY STATUS =====")
print(doda_silver["quality_status"].value_counts())

===== DODA SILVER REPORT =====
Bronze original : 48839
Lignes incomplètes : 21
Doublons exacts supprimés : 77
Lignes Silver : 48741

===== FILES =====
Silver : /content/Moroccan_Darija_LLM/data/silver/doda_silver.jsonl
Quarantine : /content/Moroccan_Darija_LLM/data/silver/doda_quarantine.jsonl

===== QUALITY STATUS =====
quality_status
pending    48741
Name: count, dtype: int64


In [72]:
import re
import pandas as pd

# Vérifier les deux colonnes sans modifier les données
def has_arabic(text):
    return bool(re.search(r"[\u0600-\u06FF]", str(text)))

def has_latin(text):
    return bool(re.search(r"[a-zA-Z]", str(text)))

doda_check = doda_silver.copy()

# 1. Détecter les écritures inattendues
doda_check["latin_has_arabic"] = (
    doda_check["darija"].apply(has_arabic)
)

doda_check["arabic_has_latin"] = (
    doda_check["darija_ar"].apply(has_latin)
)

print("===== SCRIPT CONSISTENCY =====")

print(
    "Darija Latin contenant des caractères arabes :",
    doda_check["latin_has_arabic"].sum()
)

print(
    "Darija Arabic contenant des caractères latins :",
    doda_check["arabic_has_latin"].sum()
)

# 2. Vérifier les expressions vides
print("\n===== EMPTY TEXTS =====")

for col in ["darija", "darija_ar", "eng"]:
    print(
        col,
        ":",
        (doda_check[col].fillna("").str.strip() == "").sum()
    )

# 3. Vérifier les variantes d'une même paire
pair_groups = (
    doda_check
    .groupby(["darija", "darija_ar"])["eng"]
    .nunique()
)

multiple_translations = pair_groups[pair_groups > 1]

print("\n===== TRANSLATION VARIANTS =====")
print(
    "Paires Darija identiques avec plusieurs traductions :",
    len(multiple_translations)
)

# 4. Afficher les anomalies éventuelles
print("\n===== UNEXPECTED SCRIPT EXAMPLES =====")

anomalies = doda_check[
    doda_check["latin_has_arabic"]
    | doda_check["arabic_has_latin"]
]

display(
    anomalies[
        ["darija", "darija_ar", "eng"]
    ].head(20)
)

# 5. Échantillon de 20 expressions
print("\n===== RANDOM SAMPLE =====")

display(
    doda_check[
        ["darija", "darija_ar", "eng"]
    ].sample(n=20, random_state=42)
)

===== SCRIPT CONSISTENCY =====
Darija Latin contenant des caractères arabes : 0
Darija Arabic contenant des caractères latins : 14

===== EMPTY TEXTS =====
darija : 0
darija_ar : 0
eng : 0

===== TRANSLATION VARIANTS =====
Paires Darija identiques avec plusieurs traductions : 195

===== UNEXPECTED SCRIPT EXAMPLES =====


,darija,darija_ar,eng
2135,achnahoa ATM?,أشناهوا ATM؟,Where is an ATM?
2864,jabt lik 7ta CDs dyal jazz likay3jbok,جبت ليك حتى CDs ديال الجاز ليكايعجبوك,I also got your favorite jazz CDs.
3852,wach kadan bila flmosta9bal maghadich y7tajo l...,واش كادن بلي فالمستقبل ماغاديش يحتاجو الأطفال ...,Do you think that one day in the future childr...
13170,xi 7ad tbib,xi حد طبيب,a doctor
13888,xi dwra hiya hadik,xi دورة هيا هاديك,i will take measures
13914,bhra wla xwiya,بهرا ولا xوية,he is a little bit better
14017,xkon mxa 4a3,xكون مخا عاج,when went
16499,mab9itch baghi n 7edded lik chi rendez vous 7i...,مابقيتش باغي نحدد ليك شي رانديز-vous حيت كاتجي...,i will assign you a day
23187,fin kant x7al hadi,فين كانت xحال هادي,where has he been
23544,xi joj ta3 lil,xi جوج تاع ليل,two night



===== RANDOM SAMPLE =====


,darija,darija_ar,eng
19316,3emrek teftared chi 7aja mosta7il tew9e3,عمرك تفترد شي حاجة مستحيل توقع,you should never assume it will never happen
1614,chher hada bach ma tla9ina.,شهر هادا باش ما تلاقينا.,It's been over a month since we met.
48539,ana saraha hadchi mab9ach kay3jbni,أنا صراحة هادشي مابقاش كايعجبني,Honestly i don't like this anymore
29638,oula nit lwifi 5asser,أولا نيت لويفي خاصّر,or is it brojen
36308,blati nbadal 7wayji,بلاطي نبدل حوايجي,wait me to change my clothes
18185,t3etli bezzaf bach 7essen ch3rek,تعطلي بزّاف باش حسّن شعرك,you are too late to get a cut
29985,hadchi b3id 3lik,هادشي بعيد عليك,this is far from you
3771,ki dwwzti had n8ar?,كي دوزتي هاد نهار؟,How was your day?
41620,kayn lhem ou djaj li bghiti,كاين لهم و دجاج لي بغيتي,there is meat and chicken
21345,tanfkker wach slla flw9t,تانفكّر واش سلّا فلوقت,I'm thinking if he prayed


In [73]:
# Isoler les 14 lignes contenant des caractères latins
# dans la colonne darija_ar

import re
import pandas as pd

def extract_latin_tokens(text):
    return re.findall(r"[A-Za-z]+(?:[-'][A-Za-z]+)*", str(text))

anomalies = doda_silver[
    doda_silver["darija_ar"].apply(has_latin)
].copy()

anomalies["latin_tokens"] = anomalies["darija_ar"].apply(
    lambda x: ", ".join(extract_latin_tokens(x))
)

anomalies["review_status"] = "pending"

display(
    anomalies[
        ["source_row_id", "darija", "darija_ar",
         "eng", "latin_tokens", "review_status"]
    ]
)

print("Nombre de lignes à examiner :", len(anomalies))

,source_row_id,darija,darija_ar,eng,latin_tokens,review_status
2135,2136,achnahoa ATM?,أشناهوا ATM؟,Where is an ATM?,ATM,pending
2864,2865,jabt lik 7ta CDs dyal jazz likay3jbok,جبت ليك حتى CDs ديال الجاز ليكايعجبوك,I also got your favorite jazz CDs.,CDs,pending
3852,3853,wach kadan bila flmosta9bal maghadich y7tajo l...,واش كادن بلي فالمستقبل ماغاديش يحتاجو الأطفال ...,Do you think that one day in the future childr...,PC,pending
13170,13171,xi 7ad tbib,xi حد طبيب,a doctor,xi,pending
13888,13889,xi dwra hiya hadik,xi دورة هيا هاديك,i will take measures,xi,pending
13914,13915,bhra wla xwiya,بهرا ولا xوية,he is a little bit better,x,pending
14017,14018,xkon mxa 4a3,xكون مخا عاج,when went,x,pending
16499,16500,mab9itch baghi n 7edded lik chi rendez vous 7i...,مابقيتش باغي نحدد ليك شي رانديز-vous حيت كاتجي...,i will assign you a day,vous,pending
23187,23188,fin kant x7al hadi,فين كانت xحال هادي,where has he been,x,pending
23544,23545,xi joj ta3 lil,xi جوج تاع ليل,two night,xi,pending


Nombre de lignes à examiner : 14


In [74]:
import pandas as pd
from google.colab import files

# Exporter toutes les lignes Silver DODa
review_df = doda_silver.copy()

# Ajouter des colonnes pour ta validation
review_df["validation"] = ""
review_df["correction_darija"] = ""
review_df["correction_darija_ar"] = ""
review_df["correction_eng"] = ""
review_df["commentaire"] = ""

# Organiser les colonnes pour faciliter la lecture
columns = [
    "source_row_id",
    "darija",
    "darija_ar",
    "eng",
    "validation",
    "correction_darija",
    "correction_darija_ar",
    "correction_eng",
    "commentaire"
]

review_df = review_df[columns]

# Export Excel
output_path = "/content/DODa_48741_lignes_a_verifier.xlsx"

review_df.to_excel(output_path, index=False)

print("Nombre de lignes :", len(review_df))
print("Fichier créé :", output_path)

# Télécharger le fichier
files.download(output_path)

Nombre de lignes : 48741
Fichier créé : /content/DODa_48741_lignes_a_verifier.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [75]:
import pandas as pd
from pathlib import Path

base_dir = Path("/content/Moroccan_Darija_LLM")
gold_dir = base_dir / "data" / "gold"
silver_dir = base_dir / "data" / "silver"

# 1. Charger Atlaset Gold
atlaset_path = gold_dir / "darija_daily_validated.jsonl"

# 2. Charger DODa Silver
doda_path = silver_dir / "doda_silver.jsonl"

if not atlaset_path.exists():
    raise FileNotFoundError(f"Fichier Atlaset introuvable : {atlaset_path}")

if not doda_path.exists():
    raise FileNotFoundError(f"Fichier DODa introuvable : {doda_path}")

atlaset = pd.read_json(atlaset_path, lines=True)
doda = pd.read_json(doda_path, lines=True)

# 3. Harmoniser Atlaset
atlaset_gold = pd.DataFrame({
    "darija_ar": atlaset["text"],
    "darija_latn": pd.NA,
    "english": atlaset["english"],
    "dataset_source": atlaset["dataset_source"],
    "validation_status": "user_reviewed"
})

# 4. Harmoniser DODa
doda_gold = pd.DataFrame({
    "darija_ar": doda["darija_ar"],
    "darija_latn": doda["darija"],
    "english": doda["eng"],
    "dataset_source": "DODa",
    "validation_status": "accepted_for_project"
})

# 5. Fusionner les deux sources
unified_gold = pd.concat(
    [atlaset_gold, doda_gold],
    ignore_index=True
)

# 6. Nettoyer les espaces sans modifier les expressions
for col in ["darija_ar", "darija_latn", "english"]:
    unified_gold[col] = unified_gold[col].astype("string").str.strip()
    unified_gold[col] = unified_gold[col].replace("", pd.NA)

# 7. Identifier les écritures disponibles
unified_gold["available_scripts"] = unified_gold.apply(
    lambda row: (
        "Arabic + Latin"
        if pd.notna(row["darija_ar"]) and pd.notna(row["darija_latn"])
        else "Arabic only"
    ),
    axis=1
)

# 8. Sauvegarder le corpus unifié
output_path = gold_dir / "darija_unified_language_corpus.jsonl"

unified_gold.to_json(
    output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

# 9. Rapport
print("===== UNIFIED GOLD CORPUS =====")
print("Total :", len(unified_gold))
print("\nPar source :")
print(unified_gold["dataset_source"].value_counts())

print("\nPar écriture :")
print(unified_gold["available_scripts"].value_counts())

print("\nValeurs manquantes :")
print(unified_gold[
    ["darija_ar", "darija_latn", "english"]
].isna().sum())

print("\nFichier sauvegardé :", output_path)

display(unified_gold.head(10))

===== UNIFIED GOLD CORPUS =====
Total : 50743

Par source :
dataset_source
DODa                 48741
learnmoroccan.com     1673
loecsen.com            329
Name: count, dtype: int64

Par écriture :
available_scripts
Arabic + Latin    48741
Arabic only        2002
Name: count, dtype: int64

Valeurs manquantes :
darija_ar         0
darija_latn    2002
english           0
dtype: int64

Fichier sauvegardé : /content/Moroccan_Darija_LLM/data/gold/darija_unified_language_corpus.jsonl


,darija_ar,darija_latn,english,dataset_source,validation_status,available_scripts
0,الولد مشى ل مدرسة,<NA>,The boy went to school,learnmoroccan.com,user_reviewed,Arabic only
1,الكتاب ديال ختي,<NA>,My sister's book,learnmoroccan.com,user_reviewed,Arabic only
2,أنا غادي ل الدار,<NA>,I'm going to the house,learnmoroccan.com,user_reviewed,Arabic only
3,الكتاب كاين في صاك.,<NA>,The book is in the bag.,learnmoroccan.com,user_reviewed,Arabic only
4,هداك الراجل كبير,<NA>,That man is tall,learnmoroccan.com,user_reviewed,Arabic only
5,مع صحاباتي,<NA>,With my friends (female friends),learnmoroccan.com,user_reviewed,Arabic only
6,الكتاب؟ راه هو فوق طاولة,<NA>,The book? It's on the table,learnmoroccan.com,user_reviewed,Arabic only
7,هدا الكتاب ديالو.,<NA>,This is his book.,learnmoroccan.com,user_reviewed,Arabic only
8,أنا بغيت نمشي للدار,<NA>,I want to go home,learnmoroccan.com,user_reviewed,Arabic only
9,هاد الناس ماعندهومش الحق.,<NA>,Such people have no right.,learnmoroccan.com,user_reviewed,Arabic only


In [76]:
import pandas as pd
import re

audit_df = unified_gold.copy()

# Normalisation légère uniquement pour comparer
# Les textes originaux ne seront pas modifiés.
def normalize_text(value):
    if pd.isna(value):
        return None
    value = str(value).lower().strip()
    value = re.sub(r"\s+", " ", value)
    return value

audit_df["ar_norm"] = audit_df["darija_ar"].apply(normalize_text)
audit_df["latn_norm"] = audit_df["darija_latn"].apply(normalize_text)
audit_df["eng_norm"] = audit_df["english"].apply(normalize_text)

# 1. Doublons exacts sur Darija arabe + anglais
duplicate_pairs = audit_df.duplicated(
    subset=["ar_norm", "eng_norm"],
    keep=False
)

# 2. Expressions arabes présentes dans plusieurs sources
source_count = (
    audit_df.groupby("ar_norm")["dataset_source"]
    .nunique()
)

cross_source = source_count[source_count > 1]

# 3. Expressions Franco répétées
latin_counts = audit_df["latn_norm"].value_counts()
latin_duplicates = latin_counts[latin_counts > 1]

# 4. Rapport
print("===== DUPLICATE AUDIT =====")

print("\nLignes appartenant à un doublon Darija arabe + anglais :",
      duplicate_pairs.sum())

print("Expressions arabes présentes dans plusieurs sources :",
      len(cross_source))

print("Expressions Franco répétées :",
      len(latin_duplicates))

print("\n===== CROSS-SOURCE EXAMPLES =====")

if len(cross_source) > 0:
    cross_examples = audit_df[
        audit_df["ar_norm"].isin(cross_source.index)
    ][["darija_ar", "darija_latn", "english", "dataset_source"]]

    display(cross_examples.head(20))
else:
    print("Aucun chevauchement détecté.")

print("\n===== EXACT DUPLICATE EXAMPLES =====")

display(
    audit_df[duplicate_pairs][
        ["darija_ar", "darija_latn", "english", "dataset_source"]
    ].head(20)
)

===== DUPLICATE AUDIT =====

Lignes appartenant à un doublon Darija arabe + anglais : 898
Expressions arabes présentes dans plusieurs sources : 108
Expressions Franco répétées : 238

===== CROSS-SOURCE EXAMPLES =====


,darija_ar,darija_latn,english,dataset_source
62,فين ساكن؟,<NA>,Where do you live?,learnmoroccan.com
81,الحياة زوينة,<NA>,Life is beautiful,learnmoroccan.com
85,شحال عندك من عام؟,<NA>,how old are you?,learnmoroccan.com
112,فكر مزيان,<NA>,Think carefully,learnmoroccan.com
123,أنا غادي للسوق,<NA>,I'm going to the market,learnmoroccan.com
155,شعل الضو,<NA>,Turn on the light,learnmoroccan.com
167,خليني نمشي,<NA>,Let me go,learnmoroccan.com
168,شنو سميتك؟,<NA>,What's your name?,learnmoroccan.com
211,دور على ليمن,<NA>,Turn right,learnmoroccan.com
295,سير ارتاح شوية,<NA>,Go rest a little,learnmoroccan.com



===== EXACT DUPLICATE EXAMPLES =====


,darija_ar,darija_latn,english,dataset_source
62,فين ساكن؟,<NA>,Where do you live?,learnmoroccan.com
155,شعل الضو,<NA>,Turn on the light,learnmoroccan.com
167,خليني نمشي,<NA>,Let me go,learnmoroccan.com
211,دور على ليمن,<NA>,Turn right,learnmoroccan.com
742,فين غادي؟,<NA>,Where are you going?,learnmoroccan.com
795,ديال من هاد الدار؟,<NA>,Whose house is this?,learnmoroccan.com
1126,ال,<NA>,The,learnmoroccan.com
1156,لا,<NA>,No,learnmoroccan.com
1159,غير,<NA>,Only,learnmoroccan.com
1161,وقت,<NA>,Time,learnmoroccan.com


In [77]:
# 1. Expressions arabes présentes dans plusieurs sources
cross_keys = set(cross_source.index)

cross_details = audit_df[
    audit_df["ar_norm"].isin(cross_keys)
].copy()

cross_details = cross_details.sort_values(
    ["ar_norm", "dataset_source"]
)

print("===== CROSS-SOURCE DETAILS =====")
print("Nombre d'expressions partagées :", len(cross_keys))
print("Nombre de lignes concernées :", len(cross_details))

display(
    cross_details[
        ["darija_ar", "darija_latn", "english", "dataset_source"]
    ].head(50)
)

# 2. Répartition des doublons exacts par source
exact_groups = (
    audit_df.groupby(["ar_norm", "eng_norm"], dropna=False)
    .agg(
        nombre_lignes=("dataset_source", "size"),
        sources=("dataset_source", lambda x: ", ".join(sorted(set(x)))),
        nombre_sources=("dataset_source", "nunique"),
        exemples_ar=("darija_ar", "first"),
        exemples_eng=("english", "first")
    )
    .reset_index()
)

exact_groups = exact_groups[
    exact_groups["nombre_lignes"] > 1
].sort_values("nombre_lignes", ascending=False)

print("\n===== EXACT DUPLICATE GROUPS =====")
print("Groupes avec répétitions :", len(exact_groups))
print("\nRépartition selon le nombre de sources :")
print(exact_groups["nombre_sources"].value_counts().sort_index())

display(
    exact_groups[
        [
            "exemples_ar",
            "exemples_eng",
            "nombre_lignes",
            "nombre_sources",
            "sources"
        ]
    ].head(30)
)

===== CROSS-SOURCE DETAILS =====
Nombre d'expressions partagées : 108
Nombre de lignes concernées : 315


,darija_ar,darija_latn,english,dataset_source
12471,آخر,akhor,another,DODa
1160,آخر,<NA>,Other,learnmoroccan.com
4791,آه,ah,Yeah.,DODa
4898,آه,ah,Yes,DODa
5760,آه,ah,Ah.,DODa
12100,آه,ah,yes,DODa
1471,آه,<NA>,Oh,learnmoroccan.com
12194,أكيد,akid,sure,DODa
12195,أكيد,2akid,sure,DODa
1303,أكيد,<NA>,Sure,learnmoroccan.com



===== EXACT DUPLICATE GROUPS =====
Groupes avec répétitions : 380

Répartition selon le nombre de sources :
nombre_sources
1    320
2     60
Name: count, dtype: int64


,exemples_ar,exemples_eng,nombre_lignes,nombre_sources,sources
27659,فهمت,I get it,6,2,"DODa, loecsen.com"
39691,مبروك,Congratulations,5,1,DODa
6392,اليوم,Today,5,2,"DODa, learnmoroccan.com"
21243,شهر,Month,5,2,"DODa, learnmoroccan.com"
10171,تصبح على خير,Good night,5,1,DODa
7552,بطبيعة الحال,Of course,5,1,DODa
25447,عندها,she has,5,1,DODa
7499,بصح,Really,5,2,"DODa, learnmoroccan.com"
43042,نوبتك,your turn,4,1,DODa
28567,قرن,century,4,1,DODa


In [78]:
# Comparaison des trois versions linguistiques
triple_groups = (
    audit_df.groupby(
        ["ar_norm", "latn_norm", "eng_norm"],
        dropna=False
    )
    .agg(
        nombre_lignes=("dataset_source", "size"),
        sources=("dataset_source", lambda x: ", ".join(sorted(set(x)))),
        darija_ar=("darija_ar", "first"),
        darija_latn=("darija_latn", "first"),
        english=("english", "first")
    )
    .reset_index()
)

# Garder uniquement les groupes répétés
true_duplicates = triple_groups[
    triple_groups["nombre_lignes"] > 1
].copy()

print("===== STRICT TRIPLE DUPLICATES =====")
print("Groupes strictement identiques :", len(true_duplicates))
print(
    "Lignes potentiellement supprimables :",
    int((true_duplicates["nombre_lignes"] - 1).sum())
)

print("\n===== DUPLICATE DISTRIBUTION =====")
print(
    true_duplicates["sources"].value_counts().head(15)
)

print("\n===== EXAMPLES =====")
display(
    true_duplicates[
        [
            "darija_ar",
            "darija_latn",
            "english",
            "nombre_lignes",
            "sources"
        ]
    ].head(30)
)

===== STRICT TRIPLE DUPLICATES =====
Groupes strictement identiques : 38
Lignes potentiellement supprimables : 39

===== DUPLICATE DISTRIBUTION =====
sources
DODa    38
Name: count, dtype: int64

===== EXAMPLES =====


,darija_ar,darija_latn,english,nombre_lignes,sources
174,آه,ah,Yes,2,DODa
757,أش بالّيك?,ach ballik?,what do you think?,2,DODa
2936,أنا مريض,ana mriD,I'm sick,2,DODa
2939,أنا مريضة,ana mriDa,I'm sick,2,DODa
2994,أنا مغربي,ana mghribi,I am moroccan,2,DODa
3553,أوه!,oh!,Oh!,2,DODa
4504,اجي لهنا,aji l8na,Come here,2,DODa
6123,الماكلة بنينة,lmakla bnina,The food is delicious,2,DODa
7636,بطبيعة الحال,bTabi3t l7al,Of course,3,DODa
9966,تبعني,tbe3ni,follow me,2,DODa


In [79]:
# Garder une copie des lignes qui seront retirées
duplicate_mask = unified_gold.duplicated(
    subset=["darija_ar", "darija_latn", "english"],
    keep="first"
)

removed_duplicates_df = unified_gold[duplicate_mask].copy()

# Supprimer uniquement les répétitions strictement identiques
gold_clean = unified_gold.drop_duplicates(
    subset=["darija_ar", "darija_latn", "english"],
    keep="first"
).copy()

# Sauvegarder le corpus propre
clean_path = gold_dir / "darija_unified_gold_clean.jsonl"
duplicates_path = gold_dir / "darija_removed_exact_duplicates.jsonl"

gold_clean.to_json(
    clean_path,
    orient="records",
    lines=True,
    force_ascii=False
)

removed_duplicates_df.to_json(
    duplicates_path,
    orient="records",
    lines=True,
    force_ascii=False
)

# Rapport final
print("===== GOLD DEDUPLICATION REPORT =====")
print("Avant :", len(unified_gold))
print("Doublons retirés :", len(removed_duplicates_df))
print("Après :", len(gold_clean))

print("\n===== SOURCE DISTRIBUTION =====")
print(gold_clean["dataset_source"].value_counts())

print("\n===== SCRIPT DISTRIBUTION =====")
print(gold_clean["available_scripts"].value_counts())

print("\nFichier Gold propre :", clean_path)
print("Archive des doublons :", duplicates_path)

===== GOLD DEDUPLICATION REPORT =====
Avant : 50743
Doublons retirés : 0
Après : 50743

===== SOURCE DISTRIBUTION =====
dataset_source
DODa                 48741
learnmoroccan.com     1673
loecsen.com            329
Name: count, dtype: int64

===== SCRIPT DISTRIBUTION =====
available_scripts
Arabic + Latin    48741
Arabic only        2002
Name: count, dtype: int64

Fichier Gold propre : /content/Moroccan_Darija_LLM/data/gold/darija_unified_gold_clean.jsonl
Archive des doublons : /content/Moroccan_Darija_LLM/data/gold/darija_removed_exact_duplicates.jsonl


In [80]:
# Regrouper selon les clés normalisées
normalized_groups = (
    audit_df.groupby(
        ["ar_norm", "latn_norm", "eng_norm"],
        dropna=False
    )
    .agg(
        nombre_lignes=("dataset_source", "size"),
        variantes_brutes=(
            "darija_ar",
            lambda x: " | ".join(sorted(set(x.dropna().astype(str))))
        ),
        variantes_franco=(
            "darija_latn",
            lambda x: " | ".join(sorted(set(x.dropna().astype(str))))
        ),
        traductions=(
            "english",
            lambda x: " | ".join(sorted(set(x.dropna().astype(str))))
        )
    )
    .reset_index()
)

normalized_duplicates = normalized_groups[
    normalized_groups["nombre_lignes"] > 1
].copy()

print("Groupes normalisés :", len(normalized_duplicates))
print(
    "Lignes répétées en excès :",
    int((normalized_duplicates["nombre_lignes"] - 1).sum())
)

display(
    normalized_duplicates[
        [
            "variantes_brutes",
            "variantes_franco",
            "traductions",
            "nombre_lignes"
        ]
    ].to_string(index=False)
)

Groupes normalisés : 38
Lignes répétées en excès : 39


"variantes_brutes                variantes_franco                                       traductions  nombre_lignes\n              آه                              ah                                         Yes | yes              2\n      أش بالّيك?                     ach ballik?           What do you think? | what do you think?              2\n        أنا مريض             ana mriD | ana mrid                               I'm sick | i'm sick              2\n       أنا مريضة           ana mriDa | ana mrida                               I'm sick | i'm sick              2\n       أنا مغربي                     ana mghribi                     I am Moroccan | I am moroccan              2\n            أوه!                             oh!                                         Oh! | oh!              2\n        اجي لهنا                        aji l8na                             Come here | come here              2\n   الماكلة بنينة                    lmakla bnina     The food is delicious | th

In [81]:
# Vérifier les répétitions exactes sur les données originales
raw_duplicate_mask = unified_gold.duplicated(
    subset=["darija_ar", "darija_latn", "english"],
    keep=False
)

raw_duplicates = unified_gold[raw_duplicate_mask].copy()

raw_duplicate_groups = (
    raw_duplicates.groupby(
        ["darija_ar", "darija_latn", "english"],
        dropna=False
    )
    .size()
    .reset_index(name="nombre_lignes")
    .sort_values("nombre_lignes", ascending=False)
)

print("===== RAW EXACT DUPLICATE AUDIT =====")
print("Lignes concernées :", len(raw_duplicates))
print("Groupes exacts :", len(raw_duplicate_groups))

print(
    "Lignes en excès :",
    int((raw_duplicate_groups["nombre_lignes"] - 1).sum())
    if len(raw_duplicate_groups) > 0 else 0
)

display(raw_duplicate_groups.head(30))

===== RAW EXACT DUPLICATE AUDIT =====
Lignes concernées : 0
Groupes exacts : 0
Lignes en excès : 0


,darija_ar,darija_latn,english,nombre_lignes


In [82]:
import pandas as pd
import re

df_profile = gold_clean.copy()

# 1. Calculer le nombre de mots
for col in ["darija_ar", "darija_latn", "english"]:
    df_profile[f"{col}_word_count"] = (
        df_profile[col]
        .fillna("")
        .astype(str)
        .str.split()
        .str.len()
    )

# 2. Catégoriser les textes
def classify_text(text):
    if pd.isna(text) or not str(text).strip():
        return "empty"

    text = str(text).strip()

    if "?" in text or "؟" in text:
        return "question"

    if len(text.split()) <= 4:
        return "short_expression"

    return "sentence"

df_profile["text_type"] = df_profile["darija_ar"].apply(classify_text)

# 3. Profil général
print("===== GENERAL PROFILE =====")
print("Total records:", len(df_profile))
print("\nSources:")
print(df_profile["dataset_source"].value_counts())

print("\nText types:")
print(df_profile["text_type"].value_counts())

print("\nAvailable scripts:")
print(df_profile["available_scripts"].value_counts())

# 4. Statistiques de longueur
print("\n===== WORD COUNT STATISTICS =====")
display(
    df_profile[
        ["darija_ar_word_count",
         "darija_latn_word_count",
         "english_word_count"]
    ].describe().round(2)
)

# 5. Vérifier les textes très courts
print("\n===== VERY SHORT ARABIC TEXTS (1 word) =====")
display(
    df_profile[
        df_profile["darija_ar_word_count"] == 1
    ][["darija_ar", "darija_latn", "english"]].head(20)
)

# 6. Vérifier les textes longs
print("\n===== LONGEST DARIJA EXPRESSIONS =====")
display(
    df_profile.nlargest(
        20, "darija_ar_word_count"
    )[["darija_ar", "darija_latn", "english"]]
)

===== GENERAL PROFILE =====
Total records: 50743

Sources:
dataset_source
DODa                 48741
learnmoroccan.com     1673
loecsen.com            329
Name: count, dtype: int64

Text types:
text_type
short_expression    28233
sentence            18339
question             4171
Name: count, dtype: int64

Available scripts:
available_scripts
Arabic + Latin    48741
Arabic only        2002
Name: count, dtype: int64

===== WORD COUNT STATISTICS =====


,darija_ar_word_count,darija_latn_word_count,english_word_count
count,50743.00,50743.00,50743.00
mean,4.43,4.35,5.37
std,2.31,2.46,2.55
min,1.00,0.00,1.00
25%,3.00,3.00,4.00
50%,4.00,4.00,5.00
75%,5.00,6.00,7.00
max,29.00,29.00,38.00



===== VERY SHORT ARABIC TEXTS (1 word) =====


,darija_ar,darija_latn,english
1126,ال,<NA>,The
1127,ديال,<NA>,Of
1128,ل,<NA>,To
1129,في,<NA>,In
1130,هداك,<NA>,That
1131,مع,<NA>,With
1132,هو,<NA>,It
1133,ديالو,<NA>,His
1134,أنا,<NA>,I
1135,هاد,<NA>,Such



===== LONGEST DARIJA EXPRESSIONS =====


,darija_ar,darija_latn,english
3688,"واش من نيتك, هادا راه كلب ديال الدار, فاش بغيت...","wach men niytek, hada rah kelb dial ddar, fach...","You're kidding, he's an apartment dog, when I ..."
27019,واحا تفوتها عندك خرامة و خاطبة فهاد الهوتل واح...,wa5a tfoutha 3endek 5arama ou 5atb9a fhad lhot...,after one week take hotel and take the bus the...
3975,إلا طلعنا ف هاد لهضبة غادي تولي عندنا شوفة مزي...,ila Tl3na f had lhaDaba ghadi tweli 3ndna chou...,Going up on that hill will give us the right v...
30451,لي عندو مرة وحدة هجل و لي دار لتانية دار الفال...,li 3endou mra we7da hejal ou li dar ltanya dar...,she has a girl she shoud marry her
2840,عرفت أنني كنت متردّد باش نرحل من داري حيت دوّز...,3rft annani knt mtrdded bach nr7l mn dari 7it ...,I know I was reluctant to leave my house as I ...
29423,نتا كابيتان ديال فرقتك عارف أنا هاد الماتش مهم...,nta capitaine dial fer9tek 3aref ana had lmatc...,you are the captain of the team do your best w...
3674,"داز كي لويل... صبّات عليا شتا ف طريق لخدمة, و ...","daz ki lwil... Sebbat 3lia chta f Tri9 lkhdma,...","Rubbish... I got soaked on the way to work, an..."
3376,و لكن كون ركّزتي على أنّك دير كونطرة ديال ل إس...,walakin koun rekkezti 3la annak dir contra dia...,But if you concentrated on getting a good main...
3643,مشيت ل واحد دار العجزة باش نشوف واحد صاحبي قدي...,mchit l wa7d dar l3ajaza bach nchouf wa7d sa7b...,I had to go to this retirement home and see an...
3680,غي ماتڭولش ليا بيتي تفرّج فل كورة و تسخّن شي ب...,ghi matgoulch lia biti tferrej fl kora o tsekh...,As long as you're not asking me to watch footb...


In [83]:
# Profilage par source
source_profile = (
    df_profile.groupby("dataset_source")
    .agg(
        total=("darija_ar", "count"),
        moyenne_mots_ar=("darija_ar_word_count", "mean"),
        moyenne_mots_franco=("darija_latn_word_count", "mean"),
        moyenne_mots_en=("english_word_count", "mean"),
        expressions_courtes=(
            "text_type",
            lambda x: (x == "short_expression").sum()
        ),
        phrases=(
            "text_type",
            lambda x: (x == "sentence").sum()
        ),
        questions=(
            "text_type",
            lambda x: (x == "question").sum()
        )
    )
    .round(2)
)

print("===== PROFILE BY SOURCE =====")
display(source_profile)

# Pourcentage de chaque type par source
type_distribution = pd.crosstab(
    df_profile["dataset_source"],
    df_profile["text_type"],
    normalize="index"
).mul(100).round(2)

print("\n===== TEXT TYPE DISTRIBUTION (%) =====")
display(type_distribution)

# Examiner les expressions les plus longues de chaque source
print("\n===== LONGEST EXPRESSIONS BY SOURCE =====")

for source in df_profile["dataset_source"].unique():
    print(f"\n--- {source} ---")

    display(
        df_profile[
            df_profile["dataset_source"] == source
        ].nlargest(5, "darija_ar_word_count")[
            ["darija_ar", "darija_latn", "english"]
        ]
    )

===== PROFILE BY SOURCE =====


,total,moyenne_mots_ar,moyenne_mots_franco,moyenne_mots_en,expressions_courtes,phrases,questions
dataset_source,,,,,,,
DODa,48741,4.48,4.53,5.41,26905,17843,3993
learnmoroccan.com,1673,3.40,0.00,4.39,1078,490,105
loecsen.com,329,2.48,0.00,3.45,250,6,73



===== TEXT TYPE DISTRIBUTION (%) =====


text_type,question,sentence,short_expression
dataset_source,,,
DODa,8.19,36.61,55.20
learnmoroccan.com,6.28,29.29,64.44
loecsen.com,22.19,1.82,75.99



===== LONGEST EXPRESSIONS BY SOURCE =====

--- learnmoroccan.com ---


,darija_ar,darija_latn,english
442,أنا عندي مشروع باش ندير حانوت صغير ف الدرب ديالنا,<NA>,I have a project to open a small shop in our n...
507,سمعتي أش قال الوزير ف الخطاب ديالو على السياسة...,<NA>,Did you hear what the minister said in his spe...
1018,القراية ديالك غادي يكون عندها تأثير زوين على ا...,<NA>,Your studies will have a positive impact on yo...
444,هاد العين هي المنبع ديال الما ف هاد المنطقة,<NA>,This spring is the source of water in this area
459,نجاح ديالو في العمل كان نتيجة ديال مجهود كبير.,<NA>,His success in business was the result of grea...



--- loecsen.com ---


,darija_ar,darija_latn,english
1843,واش كتفضل جوج ناموسيات ديال شخص واحد؟,<NA>,Would you prefer two single beds?
1844,واش كتفضلي جوج ناموسيات ديال شخص واحد؟,<NA>,Would you prefer two single beds?
1852,"واخا, واش يمكن ليا نحجز لهاد الليلة؟",<NA>,"OK, can I reserve for tonight?"
1854,ممكن ليك تدي ليا لباكاج ديالي عافاك؟,<NA>,"Could you take care of my luggage, please?"
1879,"ممكن نقادو لك شي حاجة, شحال بغيتي؟",<NA>,We can make it for you. How much would you lik...



--- DODa ---


,darija_ar,darija_latn,english
3688,"واش من نيتك, هادا راه كلب ديال الدار, فاش بغيت...","wach men niytek, hada rah kelb dial ddar, fach...","You're kidding, he's an apartment dog, when I ..."
27019,واحا تفوتها عندك خرامة و خاطبة فهاد الهوتل واح...,wa5a tfoutha 3endek 5arama ou 5atb9a fhad lhot...,after one week take hotel and take the bus the...
3975,إلا طلعنا ف هاد لهضبة غادي تولي عندنا شوفة مزي...,ila Tl3na f had lhaDaba ghadi tweli 3ndna chou...,Going up on that hill will give us the right v...
30451,لي عندو مرة وحدة هجل و لي دار لتانية دار الفال...,li 3endou mra we7da hejal ou li dar ltanya dar...,she has a girl she shoud marry her
2840,عرفت أنني كنت متردّد باش نرحل من داري حيت دوّز...,3rft annani knt mtrdded bach nr7l mn dari 7it ...,I know I was reluctant to leave my house as I ...


In [84]:
import pandas as pd

# Compter les expressions distinctes sans modifier les textes
for col in ["darija_ar", "darija_latn", "english"]:
    unique_count = df_profile[col].nunique(dropna=True)
    missing_count = df_profile[col].isna().sum()

    print(f"\n===== {col} =====")
    print("Total non-empty:", df_profile[col].notna().sum())
    print("Unique expressions:", unique_count)
    print("Missing values:", missing_count)

# Mesurer les expressions partagées entre les sources
print("\n===== CROSS-SOURCE DIVERSITY =====")

for col in ["darija_ar", "darija_latn"]:
    temp = df_profile.dropna(subset=[col])

    source_count = temp.groupby(col)["dataset_source"].nunique()

    print(f"\n{col}")
    print("Expressions présentes dans plusieurs sources:",
          (source_count > 1).sum())

    print("Expressions présentes dans une seule source:",
          (source_count == 1).sum())


===== darija_ar =====
Total non-empty: 50743
Unique expressions: 49732
Missing values: 0

===== darija_latn =====
Total non-empty: 48741
Unique expressions: 48483
Missing values: 2002

===== english =====
Total non-empty: 50743
Unique expressions: 47528
Missing values: 0

===== CROSS-SOURCE DIVERSITY =====

darija_ar
Expressions présentes dans plusieurs sources: 108
Expressions présentes dans une seule source: 49624

darija_latn
Expressions présentes dans plusieurs sources: 0
Expressions présentes dans une seule source: 48483


In [85]:
import pandas as pd
from pathlib import Path

# Chemin du Gold
gold_path = Path(
    "/content/Moroccan_Darija_LLM/data/gold/darija_unified_gold_clean.jsonl"
)

# Charger le corpus
gold = pd.read_json(gold_path, lines=True)

print("===== GOLD DATASET =====")
print("Total records:", len(gold))
print("\nColumns:", gold.columns.tolist())

# Vérifier les champs nécessaires
required_columns = [
    "darija_ar",
    "darija_latn",
    "english",
    "dataset_source"
]

for col in required_columns:
    print(f"{col}: {gold[col].notna().sum()} non-empty values")

# Vérifier les lignes utilisables pour chaque tâche
print("\n===== SFT TASK AVAILABILITY =====")

print(
    "Darija Arabic -> English:",
    gold.dropna(subset=["darija_ar", "english"]).shape[0]
)

print(
    "Darija Franco -> English:",
    gold.dropna(subset=["darija_latn", "english"]).shape[0]
)

print(
    "English -> Darija:",
    gold.dropna(subset=["english", "darija_ar"]).shape[0]
)

print("\n===== SAMPLE =====")
display(gold.sample(5, random_state=42))

===== GOLD DATASET =====
Total records: 50743

Columns: ['darija_ar', 'darija_latn', 'english', 'dataset_source', 'validation_status', 'available_scripts']
darija_ar: 50743 non-empty values
darija_latn: 48741 non-empty values
english: 50743 non-empty values
dataset_source: 50743 non-empty values

===== SFT TASK AVAILABILITY =====
Darija Arabic -> English: 50743
Darija Franco -> English: 48741
English -> Darija: 50743

===== SAMPLE =====


,darija_ar,darija_latn,english,dataset_source,validation_status,available_scripts
7682,أنا بصح ماكنحملش الطياران ڭاع,ana bsa7 makan7mlch Tayaran ga3,I really dont like flying at all,DODa,accepted_for_project,Arabic + Latin
34672,شكون هاد العريس جديد,chkoun had l3ris jdid,who are this new bridegroom,DODa,accepted_for_project,Arabic + Latin
11393,أنا تغلبت,ana tghlebt,I am beat,DODa,accepted_for_project,Arabic + Latin
1163,آش,None,What,learnmoroccan.com,user_reviewed,Arabic only
14554,أشنو لي بدل فلماتك و خليكوم طربحو,achno li bdel flmatc ou 5lakom treb7ou,what do you think were the turning points in t...,DODa,accepted_for_project,Arabic + Latin


In [86]:
import json
from pathlib import Path

# Dossier de sortie SFT
sft_dir = Path("/content/Moroccan_Darija_LLM/data/gold/sft")
sft_dir.mkdir(parents=True, exist_ok=True)

sft_examples = []

for _, row in gold.iterrows():

    darija_ar = str(row["darija_ar"]).strip()
    darija_latn = row["darija_latn"]
    english = str(row["english"]).strip()
    source = row["dataset_source"]

    # 1. Darija arabe -> English
    sft_examples.append({
        "messages": [
            {
                "role": "user",
                "content": f"ترجم هاد العبارة للإنجليزية: {darija_ar}"
            },
            {
                "role": "assistant",
                "content": english
            }
        ],
        "task_type": "darija_ar_to_english",
        "dataset_source": source
    })

    # 2. Darija Franco -> English
    if pd.notna(darija_latn) and str(darija_latn).strip():
        sft_examples.append({
            "messages": [
                {
                    "role": "user",
                    "content": f"Trjem had l3ibara l English: {darija_latn}"
                },
                {
                    "role": "assistant",
                    "content": english
                }
            ],
            "task_type": "darija_latn_to_english",
            "dataset_source": source
        })

    # 3. English -> Darija arabe
    sft_examples.append({
        "messages": [
            {
                "role": "user",
                "content": f"ترجم هاد الجملة للدارجة المغربية: {english}"
            },
            {
                "role": "assistant",
                "content": darija_ar
            }
        ],
        "task_type": "english_to_darija_ar",
        "dataset_source": source
    })

    # 4. English -> Darija Franco
    if pd.notna(darija_latn) and str(darija_latn).strip():
        sft_examples.append({
            "messages": [
                {
                    "role": "user",
                    "content": f"Trjem had sentence l Darija b Franco: {english}"
                },
                {
                    "role": "assistant",
                    "content": str(darija_latn).strip()
                }
            ],
            "task_type": "english_to_darija_latn",
            "dataset_source": source
        })

# Sauvegarder en JSONL
sft_path = sft_dir / "darija_translation_sft_candidates.jsonl"

with open(sft_path, "w", encoding="utf-8") as f:
    for example in sft_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

# Statistiques
sft_df = pd.DataFrame(sft_examples)

print("===== SFT GENERATION RESULTS =====")
print("Total examples:", len(sft_df))
print("\nExamples by task:")
print(sft_df["task_type"].value_counts())

print("\nExamples by source:")
print(sft_df["dataset_source"].value_counts())

print("\nSaved file:", sft_path)

print("\n===== SAMPLE SFT EXAMPLES =====")
for example in sft_examples[:4]:
    print(json.dumps(example, ensure_ascii=False, indent=2))

===== SFT GENERATION RESULTS =====
Total examples: 198968

Examples by task:
task_type
darija_ar_to_english      50743
english_to_darija_ar      50743
darija_latn_to_english    48741
english_to_darija_latn    48741
Name: count, dtype: int64

Examples by source:
dataset_source
DODa                 194964
learnmoroccan.com      3346
loecsen.com             658
Name: count, dtype: int64

Saved file: /content/Moroccan_Darija_LLM/data/gold/sft/darija_translation_sft_candidates.jsonl

===== SAMPLE SFT EXAMPLES =====
{
  "messages": [
    {
      "role": "user",
      "content": "ترجم هاد العبارة للإنجليزية: الولد مشى ل مدرسة"
    },
    {
      "role": "assistant",
      "content": "The boy went to school"
    }
  ],
  "task_type": "darija_ar_to_english",
  "dataset_source": "learnmoroccan.com"
}
{
  "messages": [
    {
      "role": "user",
      "content": "ترجم هاد الجملة للدارجة المغربية: The boy went to school"
    },
    {
      "role": "assistant",
      "content": "الولد مشى ل مدرسة"

In [87]:
# 1. Vérification des volumes
print("===== SFT DATASET PROFILE =====")
print("Total examples:", len(sft_df))

print("\n===== TASK DISTRIBUTION =====")
display(
    sft_df["task_type"]
    .value_counts()
    .rename_axis("task_type")
    .reset_index(name="count")
)

# 2. Vérifier la structure des messages
invalid_examples = []

for i, example in enumerate(sft_examples):
    messages = example.get("messages", [])

    if (
        len(messages) != 2
        or messages[0].get("role") != "user"
        or messages[1].get("role") != "assistant"
        or not messages[0].get("content", "").strip()
        or not messages[1].get("content", "").strip()
    ):
        invalid_examples.append(i)

print("\nInvalid examples:", len(invalid_examples))

# 3. Vérifier les doublons exacts (Fix: Convert messages list to a hashable string representation)
duplicate_count = sft_df.assign(
    messages_str=sft_df["messages"].apply(lambda x: str(x))
).duplicated(
    subset=["task_type", "messages_str"]
).sum()

print("Exact duplicate examples:", duplicate_count)

# 4. Vérifier les exemples par source
print("\n===== SOURCE DISTRIBUTION =====")
display(
    pd.crosstab(
        sft_df["dataset_source"],
        sft_df["task_type"]
    )
)

# 5. Afficher quelques exemples aléatoires par tâche
print("\n===== RANDOM EXAMPLES =====")

for task in sft_df["task_type"].unique():
    print(f"\n--- {task} ---")

    samples = sft_df[
        sft_df["task_type"] == task
    ].sample(
        min(2, (sft_df["task_type"] == task).sum()),
        random_state=42
    )

    for _, row in samples.iterrows():
        print("USER:", row["messages"][0]["content"])
        print("ASSISTANT:", row["messages"][1]["content"])

===== SFT DATASET PROFILE =====
Total examples: 198968

===== TASK DISTRIBUTION =====


,task_type,count
0,darija_ar_to_english,50743
1,english_to_darija_ar,50743
2,darija_latn_to_english,48741
3,english_to_darija_latn,48741



Invalid examples: 0
Exact duplicate examples: 810

===== SOURCE DISTRIBUTION =====


task_type,darija_ar_to_english,darija_latn_to_english,english_to_darija_ar,english_to_darija_latn
dataset_source,,,,
DODa,48741,48741,48741,48741
learnmoroccan.com,1673,0,1673,0
loecsen.com,329,0,329,0



===== RANDOM EXAMPLES =====

--- darija_ar_to_english ---
USER: ترجم هاد العبارة للإنجليزية: أنا بصح ماكنحملش الطياران ڭاع
ASSISTANT: I really dont like flying at all
USER: ترجم هاد العبارة للإنجليزية: شكون هاد العريس جديد
ASSISTANT: who are this new bridegroom

--- english_to_darija_ar ---
USER: ترجم هاد الجملة للدارجة المغربية: I really dont like flying at all
ASSISTANT: أنا بصح ماكنحملش الطياران ڭاع
USER: ترجم هاد الجملة للدارجة المغربية: who are this new bridegroom
ASSISTANT: شكون هاد العريس جديد

--- darija_latn_to_english ---
USER: Trjem had l3ibara l English: 3emrek teftared chi 7aja mosta7il tew9e3
ASSISTANT: you should never assume it will never happen
USER: Trjem had l3ibara l English: chher hada bach ma tla9ina.
ASSISTANT: It's been over a month since we met.

--- english_to_darija_latn ---
USER: Trjem had sentence l Darija b Franco: you should never assume it will never happen
ASSISTANT: 3emrek teftared chi 7aja mosta7il tew9e3
USER: Trjem had sentence l Darija b Franco: I

In [88]:
print("===== SFT QUALITY SUMMARY =====")

print("Total examples:", len(sft_df))

print("\nExamples per task:")
print(sft_df["task_type"].value_counts())

print("\nInvalid examples:", len(invalid_examples))
print("Exact duplicates:", duplicate_count)

print("\nExamples per source:")
print(sft_df["dataset_source"].value_counts())

print("\nSFT file size (MB):",
      round(sft_path.stat().st_size / (1024 ** 2), 2))

===== SFT QUALITY SUMMARY =====
Total examples: 198968

Examples per task:
task_type
darija_ar_to_english      50743
english_to_darija_ar      50743
darija_latn_to_english    48741
english_to_darija_latn    48741
Name: count, dtype: int64

Invalid examples: 0
Exact duplicates: 810

Examples per source:
dataset_source
DODa                 194964
learnmoroccan.com      3346
loecsen.com             658
Name: count, dtype: int64

SFT file size (MB): 48.48


In [89]:
# 1. Créer une représentation textuelle (hashable) des messages pour l'analyse
sft_df_temp = sft_df.assign(
    messages_str=sft_df["messages"].apply(lambda x: str(x))
)

# 2. Identifier les doublons exacts via la version string
duplicate_mask = sft_df_temp.duplicated(
    subset=["task_type", "messages_str"],
    keep=False
)

duplicates_df = sft_df_temp[duplicate_mask].copy()

print("===== DUPLICATE ANALYSIS =====")
print("Rows involved:", len(duplicates_df))

print("\nDuplicates by task:")
print(duplicates_df["task_type"].value_counts())

print("\nDuplicates by source:")
print(duplicates_df["dataset_source"].value_counts())

print("\n===== EXAMPLES =====")

display(
    duplicates_df[
        ["task_type", "messages", "dataset_source"]
    ].head(15)
)

# 3. Nombre de groupes de doublons
duplicate_groups = (
    duplicates_df
    .groupby(["task_type", "messages_str"], dropna=False)
    .agg(
        frequency=("dataset_source", "size"),
        sources=("dataset_source", lambda x: list(x)),
        messages=("messages", "first") # Conserver l'objet liste original pour affichage
    )
    .reset_index()
)

print("\nNumber of duplicate groups:", len(duplicate_groups))

display(
    duplicate_groups[["task_type", "messages", "frequency", "sources"]].head(10)
)

===== DUPLICATE ANALYSIS =====
Rows involved: 1402

Duplicates by task:
task_type
darija_ar_to_english      689
english_to_darija_ar      689
darija_latn_to_english     12
english_to_darija_latn     12
Name: count, dtype: int64

Duplicates by source:
dataset_source
DODa                 1378
learnmoroccan.com      12
loecsen.com            12
Name: count, dtype: int64

===== EXAMPLES =====


,task_type,messages,dataset_source
124,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",learnmoroccan.com
125,english_to_darija_ar,"[{'role': 'user', 'content': 'ترجم هاد الجملة ...",learnmoroccan.com
310,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",learnmoroccan.com
311,english_to_darija_ar,"[{'role': 'user', 'content': 'ترجم هاد الجملة ...",learnmoroccan.com
422,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",learnmoroccan.com
423,english_to_darija_ar,"[{'role': 'user', 'content': 'ترجم هاد الجملة ...",learnmoroccan.com
1484,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",learnmoroccan.com
1485,english_to_darija_ar,"[{'role': 'user', 'content': 'ترجم هاد الجملة ...",learnmoroccan.com
2774,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",learnmoroccan.com
2775,english_to_darija_ar,"[{'role': 'user', 'content': 'ترجم هاد الجملة ...",learnmoroccan.com



Number of duplicate groups: 592


,task_type,messages,frequency,sources
0,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",2,"[DODa, DODa]"
1,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",2,"[DODa, DODa]"
2,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",2,"[DODa, DODa]"
3,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",2,"[DODa, DODa]"
4,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",2,"[DODa, DODa]"
5,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",3,"[DODa, DODa, DODa]"
6,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",3,"[DODa, DODa, DODa]"
7,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",2,"[DODa, DODa]"
8,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",2,"[DODa, DODa]"
9,darija_ar_to_english,"[{'role': 'user', 'content': 'ترجم هاد العبارة...",2,"[DODa, DODa]"


In [90]:
import json
from pathlib import Path

# Copier le dataset candidat
sft_dedup = sft_df.copy()

# Créer une clé unique à partir des messages
sft_dedup["_example_key"] = sft_dedup["messages"].apply(
    lambda x: json.dumps(x, ensure_ascii=False, sort_keys=True)
)

# Regrouper les sources de chaque exemple
source_mapping = (
    sft_dedup.groupby("_example_key")["dataset_source"]
    .agg(lambda x: sorted(set(x)))
    .to_dict()
)

# Garder une seule occurrence de chaque exemple
sft_dedup = sft_dedup.drop_duplicates(
    subset=["_example_key"],
    keep="first"
).copy()

# Conserver toutes les sources
sft_dedup["dataset_sources"] = (
    sft_dedup["_example_key"].map(source_mapping)
)

# Calculer les doublons restants sur la clé hashable avant de la supprimer
remaining_duplicates_count = sft_dedup.duplicated(
    subset=["task_type", "_example_key"]
).sum()

# Supprimer la colonne technique
sft_dedup = sft_dedup.drop(columns=["_example_key"])

# Sauvegarder
dedup_path = Path(
    "/content/Moroccan_Darija_LLM/data/gold/sft/"
    "darija_translation_sft_deduplicated.jsonl"
)

with open(dedup_path, "w", encoding="utf-8") as f:
    for record in sft_dedup.to_dict(orient="records"):
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("===== DEDUPLICATION RESULTS =====")
print("Before:", len(sft_df))
print("After:", len(sft_dedup))
print("Removed:", len(sft_df) - len(sft_dedup))
print("Remaining duplicates:", remaining_duplicates_count)

print("\n===== TASK DISTRIBUTION =====")
print(sft_dedup["task_type"].value_counts())

print("\nSaved:", dedup_path)

===== DEDUPLICATION RESULTS =====
Before: 198968
After: 198158
Removed: 810
Remaining duplicates: 0

===== TASK DISTRIBUTION =====
task_type
darija_ar_to_english      50344
english_to_darija_ar      50344
darija_latn_to_english    48735
english_to_darija_latn    48735
Name: count, dtype: int64

Saved: /content/Moroccan_Darija_LLM/data/gold/sft/darija_translation_sft_deduplicated.jsonl


In [91]:
%pip install --no-deps --force-reinstall \
    "google-genai==2.34.0" \
    "google-auth==2.49.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.5 MB/s  0:00:00
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [google-auth]


In [92]:
import importlib.metadata as md

for package in ["google-genai", "google-auth", "torch", "transformers", "peft", "trl"]:
    print(f"{package}: {md.version(package)}")

google-genai: 2.34.0
google-auth: 2.49.0
torch: 2.14.1+cpu
transformers: 5.22.2
peft: 0.21.2
trl: 1.17.1


In [93]:
from google.colab import userdata
from google import genai
import time

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="جاوب بالدارجة المغربية: سلام، كيداير اليوم؟"
)

print("Gemini connecté avec succès !")
print(response.text)

Gemini connecté avec succès !
وعليكم السلام! الحمد لله، كولشي بيخير وعلى خير، ربي يحفظك. 

ونتا كيداير اليوم؟ كولشي مزيان؟


In [94]:
import json
import time
import re

def extract_json(text):
    """Extrait un objet JSON de la réponse Gemini."""
    text = text.strip()

    # Retirer les éventuelles balises Markdown
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # Chercher le début de l'objet
    start = text.find("{")

    if start == -1:
        raise ValueError("Aucun objet JSON trouvé.")

    # Extraction équilibrée des accolades,
    # en respectant les chaînes de caractères.
    depth = 0
    in_string = False
    escaped = False

    for i in range(start, len(text)):
        char = text[i]

        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
        else:
            if char == '"':
                in_string = True
            elif char == "{":
                depth += 1
            elif char == "}":
                depth -= 1

                if depth == 0:
                    return json.loads(text[start:i+1])

    raise ValueError("JSON incomplet : accolades non fermées.")


def validate_conversation(data, expected_style):

    assert isinstance(data, dict)
    assert data.get("script_style") == expected_style

    messages = data.get("messages")

    assert isinstance(messages, list)
    assert 6 <= len(messages) <= 10
    assert len(messages) % 2 == 0

    for i, message in enumerate(messages):
        expected_role = "user" if i % 2 == 0 else "assistant"

        assert message.get("role") == expected_role
        assert isinstance(message.get("content"), str)
        assert message["content"].strip()

    full_text = " ".join(m["content"] for m in messages)

    has_arabic = bool(re.search(r"[\u0600-\u06FF]", full_text))
    has_latin = bool(re.search(r"[a-zA-Z]", full_text))

    if expected_style == "arabic":
        assert has_arabic and not has_latin

    elif expected_style == "franco":
        assert has_latin and not has_arabic

    elif expected_style == "mixed":
        assert has_arabic and has_latin

    return True


def generate_conversation_v2(scenario, max_attempts=3):

    prompt = build_prompt(
        scenario["style"],
        scenario["instruction"],
        scenario["topic"]
    )

    for attempt in range(1, max_attempts + 1):

        try:
            response = client.models.generate_content(
                model=generation_config["model"],
                contents=prompt,
                config={
                    "temperature": generation_config["temperature"],
                    "max_output_tokens": 3000,
                    "response_mime_type": "application/json"
                }
            )

            if not response.text:
                raise ValueError("Réponse Gemini vide.")

            data = extract_json(response.text)

            validate_conversation(data, scenario["style"])

            data["generation_source"] = "Gemini"
            data["quality_status"] = "pending_human_review"

            return data

        except Exception as e:
            print(f"Tentative {attempt}/{max_attempts} : {e}")

            if attempt == max_attempts:
                raise

            time.sleep(5)

In [97]:
# Initialisation du dictionnaire si non défini
if 'generation_config' not in globals():
    generation_config = {}

generation_config["model"] = "gemini-3.8-flash"

print("Modèle sélectionné :", generation_config["model"])

Modèle sélectionné : gemini-3.8-flash


In [98]:
# Synchroniser la configuration avec le modèle testé avec succès
generation_config = {
    "model": "gemini-3.8-flash"
}

print("Configuration actuelle :", generation_config)

Configuration actuelle : {'model': 'gemini-3.8-flash'}


In [99]:
print("scenarios existe :", "scenarios" in globals())
print("generate_conversation_v2 existe :", "generate_conversation_v2" in globals())
print("generation_config :", generation_config)

scenarios existe : False
generate_conversation_v2 existe : True
generation_config : {'model': 'gemini-3.8-flash'}


In [100]:
scenarios = [
    {
        "id": "pilot_ar_001",
        "style": "arabic",
        "topic": "opportunity_and_career",
        "situation": (
            "شخص توصل بعرض خدمة جديد، ولكن متردد واش يقبلو "
            "حيت خايف من التغيير وباغي ياخذ رأي صاحبو."
        )
    },
    {
        "id": "pilot_latn_001",
        "style": "franco",
        "topic": "friendship_and_misunderstanding",
        "situation": (
            "Jouj s7ab tkhassmo بسبب سوء تفاهم، "
            "wa7ed fihom bgha ysali lmochkil w ytsala7 m3a lakhor."
        )
    },
    {
        "id": "pilot_mixed_001",
        "style": "mixed",
        "topic": "study_and_exam",
        "situation": (
            "Wa7ed talib kaywjed l exam, 3endo stress "
            "w kayhder m3a sa7bo 3la طريقة تنظيم المراجعة."
        )
    }
]

for scenario in scenarios:
    print(scenario["id"], "|", scenario["style"], "|", scenario["topic"])

pilot_ar_001 | arabic | opportunity_and_career
pilot_latn_001 | franco | friendship_and_misunderstanding
pilot_mixed_001 | mixed | study_and_exam


In [101]:
def generate_conversation_v2(scenario, max_attempts=3):

    # Récupérer les informations du scénario
    style = scenario["style"]
    topic = scenario["topic"]
    situation = scenario["situation"]

    # Construire le prompt dans le bon ordre
    prompt = build_prompt(
        style=style,
        topic=topic,
        situation=situation
    )

    for attempt in range(1, max_attempts + 1):

        try:
            response = client.models.generate_content(
                model=generation_config["model"],
                contents=prompt,
                config={
                    "temperature": generation_config["temperature"],
                    "max_output_tokens": generation_config["max_output_tokens"],
                    "response_mime_type": "application/json"
                }
            )

            if not response.text:
                raise ValueError("Réponse Gemini vide.")

            # Extraire le JSON
            data = extract_json(response.text)

            # Vérifier le format
            if data.get("script_style") != style:
                raise ValueError("Style retourné incorrect.")

            validate_conversation(data, style)

            # Ajouter les métadonnées
            data["conversation_id"] = scenario["id"]
            data["topic"] = topic
            data["generation_source"] = "Gemini"
            data["quality_status"] = "pending_human_review"

            return data

        except Exception as e:

            print(f"Tentative {attempt}/{max_attempts} : {e}")

            if attempt == max_attempts:
                raise

            time.sleep(5)

In [104]:
# Compléter la configuration globale requise par generate_conversation_v2
if 'generation_config' not in globals():
    generation_config = {}

generation_config.update({
    "model": "gemini-3.8-flash",
    "temperature": 0.7,
    "max_output_tokens": 1024
})

def build_prompt(style, topic, situation):
    """Génère le prompt système pour guider la génération de conversation."""
    style_instructions = {
        "arabic": "en alphabet arabe uniquement (pas de caractères latins).",
        "franco": "en Darija latine (Franco-arabe / Arabizi) uniquement, en utilisant des chiffres comme 3, 7, 9 pour les lettres arabes correspondantes.",
        "mixed": "en combinant de manière naturelle l'alphabet arabe et l'alphabet latin (Franco-arabe) au sein de la même discussion."
    }

    instruction = style_instructions.get(style, "en Darija marocain.")

    prompt = f"""Rôle : Vous êtes un générateur de conversations réalistes en Darija marocain (dialecte marocain).

Tâche : Générez une conversation naturelle et fluide entre deux amis proches à propos du sujet suivant :
Sujet : {topic}
Situation spécifique : {situation}

Style d'écriture : Vous devez rédiger {instruction}

Format de sortie obligatoire : Vous devez répondre UNIQUEMENT sous la forme d'un dictionnaire JSON valide contenant exactement ces deux clés :
- \"script_style\": \"{style}\"
- \"messages\": une liste de 6 à 10 messages alternant entre \"user\" et \"assistant\". Chaque message est un dictionnaire avec \"role\" (\"user\" ou \"assistant\") et \"content\" (le texte en Darija).

Exemple de format JSON attendu :
{{
  \"script_style\": \"{style}\",
  \"messages\": [
    {{\"role\": \"user\", \"content\": \"...\"}},
    {{\"role\": \"assistant\", \"content\": \"...\"}}
  ]
}}
"""
    return prompt

# Exécution du test maintenant que build_prompt et generation_config sont correctement configurés
test_conversation = generate_conversation_v2(scenarios[0])

print("Style :", test_conversation["script_style"])
print("Nombre de messages :", len(test_conversation["messages"]))
print("Statut :", test_conversation["quality_status"])

print("\n--- Conversation ---")
for message in test_conversation["messages"]:
    print(f'{message["role"]}: {message["content"]}')

Tentative 1/3 : JSON incomplet : accolades non fermées.
Tentative 2/3 : JSON incomplet : accolades non fermées.
Tentative 3/3 : 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 9h11m35.478636008s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'v

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 9h11m35.478636008s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.g

In [105]:
import pandas as pd
import json
from pathlib import Path

BASE = Path("/content/Moroccan_Darija_LLM")

translation_path = BASE / "data/gold/sft/darija_translation_sft_deduplicated.jsonl"

translations = pd.read_json(translation_path, lines=True)

print("Exemples de traduction :", len(translations))
print("\nRépartition :")
print(translations["task_type"].value_counts())

print("\nColonnes :", translations.columns.tolist())

Exemples de traduction : 198158

Répartition :
task_type
darija_ar_to_english      50344
english_to_darija_ar      50344
darija_latn_to_english    48735
english_to_darija_latn    48735
Name: count, dtype: int64

Colonnes : ['messages', 'task_type', 'dataset_source', 'dataset_sources']


In [106]:
# Examiner 4 exemples, un par type de tâche

for task in translations["task_type"].unique():
    sample = translations[translations["task_type"] == task].iloc[0]

    print("\n" + "=" * 60)
    print("TASK :", task)
    print("=" * 60)

    print(json.dumps(
        sample["messages"],
        ensure_ascii=False,
        indent=2
    ))


TASK : darija_ar_to_english
[
  {
    "role": "user",
    "content": "ترجم هاد العبارة للإنجليزية: الولد مشى ل مدرسة"
  },
  {
    "role": "assistant",
    "content": "The boy went to school"
  }
]

TASK : english_to_darija_ar
[
  {
    "role": "user",
    "content": "ترجم هاد الجملة للدارجة المغربية: The boy went to school"
  },
  {
    "role": "assistant",
    "content": "الولد مشى ل مدرسة"
  }
]

TASK : darija_latn_to_english
[
  {
    "role": "user",
    "content": "Trjem had l3ibara l English: homa mkhbbyin chi haja, ana mti99en!"
  },
  {
    "role": "assistant",
    "content": "They're hiding something, I'm sure!"
  }
]

TASK : english_to_darija_latn
[
  {
    "role": "user",
    "content": "Trjem had sentence l Darija b Franco: They're hiding something, I'm sure!"
  },
  {
    "role": "assistant",
    "content": "homa mkhbbyin chi haja, ana mti99en!"
  }
]


In [107]:
import re
import json
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

# 1. Extraire la phrase anglaise pour regrouper les traductions
def normalize_text(text):
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def get_english_text(row):
    user_text = row["messages"][0]["content"]
    assistant_text = row["messages"][1]["content"]

    if row["task_type"] in [
        "darija_ar_to_english",
        "darija_latn_to_english"
    ]:
        return normalize_text(assistant_text)
    else:
        return normalize_text(user_text.split(":", 1)[-1])

translations["group_id"] = translations.apply(get_english_text, axis=1)

# 2. Séparer les groupes : aucune phrase anglaise commune
# entre train et validation
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.05,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(translations, groups=translations["group_id"])
)

train_pool = translations.iloc[train_idx].copy()
val_pool = translations.iloc[val_idx].copy()

# 3. Sélectionner 5 000 exemples par tâche pour le premier entraînement
train_parts = []
val_parts = []

for task in translations["task_type"].unique():
    task_train = train_pool[train_pool["task_type"] == task]
    task_val = val_pool[val_pool["task_type"] == task]

    train_parts.append(
        task_train.sample(n=min(5000, len(task_train)), random_state=42)
    )
    val_parts.append(
        task_val.sample(n=min(500, len(task_val)), random_state=42)
    )

train_df = pd.concat(train_parts).sample(frac=1, random_state=42).reset_index(drop=True)
val_df = pd.concat(val_parts).sample(frac=1, random_state=42).reset_index(drop=True)

# 4. Garder uniquement le format messages
train_data = [{"messages": row} for row in train_df["messages"]]
val_data = [{"messages": row} for row in val_df["messages"]]

# 5. Sauvegarder les fichiers
output_dir = Path("/content/Moroccan_Darija_LLM/data/gold/sft")
output_dir.mkdir(parents=True, exist_ok=True)

train_path = output_dir / "train.jsonl"
val_path = output_dir / "validation.jsonl"

for path, data in [(train_path, train_data), (val_path, val_data)]:
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("TRAIN :", len(train_data))
print("VALIDATION :", len(val_data))
print("\nRépartition TRAIN :")
print(train_df["task_type"].value_counts())
print("\nRépartition VALIDATION :")
print(val_df["task_type"].value_counts())
print("\nFichiers sauvegardés :")
print(train_path)
print(val_path)

TRAIN : 20000
VALIDATION : 2000

Répartition TRAIN :
task_type
darija_latn_to_english    5000
darija_ar_to_english      5000
english_to_darija_ar      5000
english_to_darija_latn    5000
Name: count, dtype: int64

Répartition VALIDATION :
task_type
english_to_darija_latn    500
darija_ar_to_english      500
darija_latn_to_english    500
english_to_darija_ar      500
Name: count, dtype: int64

Fichiers sauvegardés :
/content/Moroccan_Darija_LLM/data/gold/sft/train.jsonl
/content/Moroccan_Darija_LLM/data/gold/sft/validation.jsonl


In [108]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

# Tester le format conversationnel avec le bon nom de variable (train_df)
sample_messages = train_df.iloc[0]["messages"]

formatted_text = tokenizer.apply_chat_template(
    sample_messages,
    tokenize=False,
    add_generation_prompt=False
)

print("===== FORMAT QWEN =====")
print(formatted_text)

print("\n===== TOKENIZATION =====")
tokens = tokenizer(formatted_text)

print("Nombre de tokens :", len(tokens["input_ids"]))
print("EOS token :", tokenizer.eos_token)
print("PAD token :", tokenizer.pad_token)

===== FORMAT QWEN =====
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Trjem had l3ibara l English: jawbini dghiya darouri<|im_end|>
<|im_start|>assistant
answer me please<|im_end|>


===== TOKENIZATION =====
Nombre de tokens : 52
EOS token : <|im_end|>
PAD token : <|endoftext|>


In [109]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Configuration quantification 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Chargement du modèle
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Modèle chargé :", MODEL_NAME)
print("VRAM utilisée :", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("GPU :", torch.cuda.get_device_name(0))
print("Paramètres entraînables avant LoRA :", sum(p.numel() for p in model.parameters() if p.requires_grad))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Modèle chargé : Qwen/Qwen2.5-1.5B-Instruct
VRAM utilisée : 0.0 GB


AssertionError: Torch not compiled with CUDA enabled

In [110]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

# Désactiver le cache pendant l'entraînement
model.config.use_cache = False

# Préparation QLoRA
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

# Configuration LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

# Ajouter les adaptateurs
model = get_peft_model(model, lora_config)

# Vérifier les paramètres
model.print_trainable_parameters()

print("\n===== GPU =====")
print("VRAM utilisée :", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("QLoRA prêt !")

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

===== GPU =====
VRAM utilisée : 0.0 GB
QLoRA prêt !


In [111]:
from datasets import load_dataset

# Charger les fichiers préparés
dataset = load_dataset(
    "json",
    data_files={
        "train": str(BASE / "data/gold/sft/train.jsonl"),
        "validation": str(BASE / "data/gold/sft/validation.jsonl")
    }
)

# Transformer les conversations avec le template Qwen
def tokenize_function(batch):
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        for messages in batch["messages"]
    ]

    return tokenizer(
        texts,
        truncation=True,
        max_length=256
    )

# Tokenisation
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Train :", len(tokenized_dataset["train"]))
print("Validation :", len(tokenized_dataset["validation"]))

print("\nExemple tokenisé :")
print(tokenized_dataset["train"][0])

print("\nNombre de tokens du premier exemple :",
      len(tokenized_dataset["train"][0]["input_ids"]))

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Train : 20000
Validation : 2000

Exemple tokenisé :
{'input_ids': [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553, 54364, 14817, 13, 1446, 525, 264, 10950, 17847, 13, 151645, 198, 151644, 872, 198, 1282, 62903, 1030, 326, 18, 579, 5059, 326, 6364, 25, 16535, 65, 6591, 294, 75076, 7755, 15269, 21467, 151645, 198, 151644, 77091, 198, 9217, 752, 4486, 151645, 198], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Nombre de tokens du premier exemple : 52


In [112]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

# Préparation des batches
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8
)

# Paramètres d'entraînement
training_args = TrainingArguments(
    output_dir=str(BASE / "models/qwen-darija-checkpoints"),

    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="cosine",

    fp16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",

    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,

    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none"
)

# Création du Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

print("Trainer configuré !")
print("Nombre d'exemples :", len(tokenized_dataset["train"]))
print("Batch effectif :", 4 * 4)
print("Epochs :", training_args.num_train_epochs)

Trainer configuré !
Nombre d'exemples : 20000
Batch effectif : 16
Epochs : 1


In [113]:
# Lancer le fine-tuning QLoRA
train_result = trainer.train()

print("\n===== RÉSULTATS DE L'ENTRAÎNEMENT =====")
print("Training loss :", train_result.training_loss)
print("Durée :", train_result.metrics["train_runtime"], "secondes")
print("Échantillons / seconde :", train_result.metrics["train_samples_per_second"])

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
print("===== ÉTAT DU FINE-TUNING =====")

print("Global step :", trainer.state.global_step)
print("Étapes prévues :", trainer.state.max_steps)
print("Epoch :", trainer.state.epoch)

print("\n===== HISTORIQUE =====")
print("Nombre de logs :", len(trainer.state.log_history))

for log in trainer.state.log_history[-10:]:
    print(log)

print("\n===== GPU =====")
print("VRAM :", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

In [ ]:
from pathlib import Path

ADAPTER_PATH = BASE / "models/qwen-darija-lora"

ADAPTER_PATH.mkdir(parents=True, exist_ok=True)

model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("Modèle sauvegardé !")
print("Chemin :", ADAPTER_PATH)
print("Fichiers :")

for file in ADAPTER_PATH.iterdir():
    print("-", file.name)

In [ ]:
from pathlib import Path

ADAPTER_PATH = Path("/content/Moroccan_Darija_LLM/models/qwen-darija-lora")

print("Dossier existe :", ADAPTER_PATH.exists())
print("Dossier valide :", ADAPTER_PATH.is_dir())

if ADAPTER_PATH.exists():
    print("\nContenu :")
    for file in ADAPTER_PATH.iterdir():
        print("-", file.name)
else:
    print("\nDossier parent existe :",
          ADAPTER_PATH.parent.exists())

In [ ]:
from pathlib import Path

print("===== ENVIRONNEMENT COLAB =====")

print("Dossier /content :", list(Path("/content").iterdir()))

print("\n===== GOOGLE DRIVE =====")

drive_path = Path("/content/drive/MyDrive")

print("Drive monté :", drive_path.exists())

if drive_path.exists():
    print("Dossiers disponibles :")
    for item in drive_path.iterdir():
        print("-", item.name)

print("\n===== VARIABLES EN MÉMOIRE =====")

for name in ["model", "trainer", "tokenizer", "base_model"]:
    print(name, ":", name in globals())

In [ ]:
import torch
import psutil

print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("RAM disponible :", round(psutil.virtual_memory().available / 1024**3, 2), "GB")

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("VRAM :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

In [ ]:
import importlib.metadata as md

packages = [
    "transformers",
    "datasets",
    "peft",
    "trl",
    "accelerate",
    "psutil"
]

for package in packages:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NON INSTALLÉ")

In [ ]:
%pip install -U trl

In [ ]:
import torch
import transformers
import peft
import trl

print("PyTorch :", torch.__version__)
print("Transformers :", transformers.__version__)
print("PEFT :", peft.__version__)
print("TRL :", trl.__version__)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
)

model.config.use_cache = False

print("Modèle chargé avec succès !")
print("Paramètres :", sum(p.numel() for p in model.parameters()))
print("Device :", next(model.parameters()).device)

In [ ]:
model.eval()

messages = [
    {
        "role": "user",
        "content": "شنو كتدير اليوم؟ جاوبني بالدارجة المغربية."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

input_ids = inputs["input_ids"]

import time
start = time.time()

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=60,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][input_ids.shape[1]:],
    skip_special_tokens=True
)

print("Question :", messages[0]["content"])
print("Réponse :", response)
print("Temps :", round(time.time() - start, 2), "secondes")

In [ ]:
import json

mini_data = [
    ("شنو سميتك؟", "What is your name?"),
    ("فين ساكن؟", "Where do you live?"),
    ("بغيت نشري الخبز", "I want to buy bread."),
    ("واش نقدر ندخل؟", "Can I come in?"),
    ("ما فهمتش", "I didn't understand."),
    ("غادي نمشي للسوق", "I'm going to the market."),
    ("عافاك عطيني كاس ديال الما", "Please give me a glass of water."),
    ("اليوم الجو زوين", "The weather is nice today."),
    ("شحال هادي؟", "How much is this?"),
    ("كنقرا فالجامعة", "I study at university."),
    ("تسنى شوية", "Wait a little."),
    ("بغيت نتعلم الإنجليزية", "I want to learn English."),
    ("واش كلشي بخير؟", "Is everything okay?"),
    ("ما عنديش الوقت دابا", "I don't have time right now."),
    ("غدا غادي نخدم", "Tomorrow I am going to work."),
    ("شكرا بزاف على المساعدة", "Thank you very much for the help.")
]

mini_dataset = []

for darija, english in mini_data:
    mini_dataset.append({
        "messages": [
            {
                "role": "user",
                "content": f"ترجم هاد العبارة للإنجليزية: {darija}"
            },
            {
                "role": "assistant",
                "content": english
            }
        ]
    })

with open("mini_train.jsonl", "w", encoding="utf-8") as f:
    for item in mini_dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Nombre d'exemples :", len(mini_dataset))
print("Fichier créé : mini_train.jsonl")
print("\nPremier exemple :")
print(json.dumps(mini_dataset[0], ensure_ascii=False, indent=2))

In [ ]:
# Désinstaller la version incompatible de torchao pour éviter le conflit avec PEFT
!pip uninstall -y torchao

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model.config.use_cache = False

model = get_peft_model(model, lora_config)

print("Type :", type(model))
model.print_trainable_parameters()

In [ ]:
from transformers import (
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./qwen-darija-cpu-test",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    use_cpu=True,
    dataloader_num_workers=0
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer préparé !")
print("Nombre d'exemples :", len(tokenized_dataset))
print("Nombre d'étapes :", len(trainer.get_train_dataloader()))

In [ ]:
import time
import psutil

start_time = time.time()

print("Début du fine-tuning CPU...")

train_result = trainer.train()

elapsed = time.time() - start_time

print("\n===== TRAINING RESULTS =====")
print("Durée :", round(elapsed / 60, 2), "minutes")
print("Train loss :", train_result.training_loss)
print("RAM disponible :", round(
    psutil.virtual_memory().available / 1024**3, 2
), "GB")

In [ ]:
import os
import shutil

SAVE_DIR = "./qwen-darija-lora-cpu-test"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

ZIP_PATH = shutil.make_archive(
    SAVE_DIR,
    "zip",
    root_dir=SAVE_DIR
)

print("Adaptateur sauvegardé :", SAVE_DIR)
print("Archive créée :", ZIP_PATH)
print("Taille :", round(os.path.getsize(ZIP_PATH) / 1024**2, 2), "MB")
print("Fichiers :", os.listdir(SAVE_DIR))

In [ ]:
import time
import torch

messages = [
    {
        "role": "user",
        "content": "شنو كتدير اليوم؟ جاوبني بالدارجة المغربية."
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

input_ids = inputs["input_ids"]

def generate_response():
    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids,
            max_new_tokens=40,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(
        output[0][input_ids.shape[1]:],
        skip_special_tokens=True
    )

# AVANT : désactiver temporairement LoRA
start = time.time()

with model.disable_adapter():
    response_before = generate_response()

time_before = time.time() - start

# APRÈS : réactiver LoRA
start = time.time()

response_after = generate_response()

time_after = time.time() - start

print("===== AVANT FINE-TUNING =====")
print(response_before)
print("Temps :", round(time_before, 2), "secondes")

print("\n===== APRÈS FINE-TUNING =====")
print(response_after)
print("Temps :", round(time_after, 2), "secondes")

In [ ]:
test_cases = [
    ("فين مشيتي البارح؟", "Where did you go yesterday?"),
    ("بغيت نمشي نشري شي حوايج", "I want to go buy some things."),
    ("علاش ما عيطتيش ليا؟", "Why didn't you call me?"),
    ("راني عيان اليوم", "I'm tired today."),
    ("غادي نرجع من بعد", "I'll come back later.")
]
import pandas as pd
import torch

def translate_darija(text):
    messages = [
        {
            "role": "user",
            "content": f"ترجم هاد العبارة للإنجليزية: {text}"
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    input_ids = inputs["input_ids"]

    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids,
            max_new_tokens=50,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        output[0][input_ids.shape[1]:],
        skip_special_tokens=True
    )


results = []

for darija, reference in test_cases:

    with model.disable_adapter():
        baseline = translate_darija(darija)

    fine_tuned = translate_darija(darija)

    results.append({
        "Darija": darija,
        "Reference": reference,
        "Base model": baseline,
        "Fine-tuned model": fine_tuned
    })

df_results = pd.DataFrame(results)

display(df_results)

In [ ]:
test_prompt = "شنو سميتك؟"

messages = [
    {
        "role": "user",
        "content": f"ترجم هاد العبارة للإنجليزية: {test_prompt}"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

with torch.inference_mode():
    output = model.generate(
        input_ids=inputs["input_ids"],
        max_new_tokens=30,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
))

In [41]:
import os

print("Dossier courant :", os.getcwd())
print("Contenu de /content :")
print(os.listdir("/content"))

Dossier courant : /content
Contenu de /content :
['.config', 'mini_train.jsonl', 'Moroccan_Darija_LLM', 'darija_llm_project', 'qwen-darija-lora-cpu-test', 'qwen-darija-cpu-test', 'qwen-darija-lora-cpu-test.zip', 'sample_data']


### Étape 1 : Génération du fichier `README.md` pour le dépôt GitHub

In [114]:
readme_content = """# Moroccan Darija LLM (Qwen-2.5-1.5B-Instruct Fine-Tuning)

Ce projet implémente un pipeline complet d'ingénierie de données et d'entraînement (Fine-Tuning LoRA / SFT) pour adapter le modèle de langage **Qwen-2.5-1.5B-Instruct** à la traduction et la génération en **Darija Marocaine** (arabe marocain), couvrant à la fois l'écriture arabe classique et le format Franco-Arabe (Arabizi).

## 📊 Vue d'ensemble du Dataset (Gold Standard)

Notre dataset unifié et nettoyé rassemble **50 743 paires linguistiques** uniques :
- **DODa** : 48 741 exemples (Arabe + Écriture Franco/Latin).
- **Atlaset** (filtré et validé) : 2 002 exemples de la vie quotidienne.

### Tâches d'entraînement (SFT)
À partir de ce corpus, nous avons généré un pipeline de **198 158 tâches d'instruction** réparties comme suit :
1. **Traductions Darija Arabe ➔ Anglais** : 50 344 exemples.
2. **Traductions Anglais ➔ Darija Arabe** : 50 344 exemples.
3. **Traductions Darija Franco ➔ Anglais** : 48 735 exemples.
4. **Traductions Anglais ➔ Darija Franco** : 48 735 exemples.

Le dataset d'entraînement final comprend **20 000 échantillons équilibrés** (`train.jsonl`) et un ensemble de validation de **2 000 échantillons** (`validation.jsonl`).

## 📁 Structure du Projet

```directory
Moroccan_Darija_LLM/
├── data/
│   ├── bronze/       # Données brutes de départ (Atlaset & DODa)
│   ├── silver/       # Données après nettoyage, normalisation et déduplication
│   └── gold/         # Dataset final unifié et fichiers SFT (train.jsonl, validation.jsonl)
├── models/           # Checkpoints locaux et adaptateurs LoRA sauvegardés
├── src/              # Scripts d'exécution principaux
│   ├── train.py      # Script de fine-tuning QLoRA de production
│   └── push.py       # Sauvegarde automatique sur Hugging Face Hub
├── requirements.txt  # Dépendances requises
└── README.md         # Présentation et documentation du projet
```

## 🚀 Installation & Utilisation

### 1. Cloner le dépôt et installer les dépendances
```bash
git clone https://github.com/votre-username/Moroccan-Darija-LLM.git
cd Moroccan-Darija-LLM
pip install -r requirements.txt
```

### 2. Lancer l'entraînement (Fine-Tuning LoRA)
Pour lancer l'entraînement de l'adaptateur sur GPU (recommandé : T4, A100, etc.) :
```bash
python src/train.py --model_name "Qwen/Qwen2.5-1.5B-Instruct" --train_file "data/gold/sft/train.jsonl" --val_file "data/gold/sft/validation.jsonl"
```
"""

import os
from pathlib import Path

project_root = Path("/content/Moroccan_Darija_LLM")
src_dir = project_root / "src"
src_dir.mkdir(parents=True, exist_ok=True)

with open(project_root / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_content.strip())

print("README.md créé avec succès dans", project_root)

README.md créé avec succès dans /content/Moroccan_Darija_LLM


### Étape 2 : Génération du fichier `requirements.txt`

In [115]:
requirements_content = """
transformers>=4.40.0
datasets>=2.19.0
peft>=0.10.0
trl>=0.8.6
accelerate>=0.30.0
bitsandbytes>=0.43.0
sentencepiece
protobuf
pandas
torch>=2.1.0
scikit-learn
"""

with open(project_root / "requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements_content.strip())

print("requirements.txt créé avec succès dans", project_root)

requirements.txt créé avec succès dans /content/Moroccan_Darija_LLM


### Étape 3 : Création du script d'entraînement robuste `src/train.py`

In [116]:
train_script_content = """# -*- coding: utf-8 -*-
import argparse
import os
from pathlib import Path
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

def parse_args():
    parser = argparse.ArgumentParser(description="Fine-tuning Qwen-2.5 sur la Darija marocaine.")
    parser.add_argument("--model_name", type=str, default="Qwen/Qwen2.5-1.5B-Instruct")
    parser.add_argument("--train_file", type=str, required=True)
    parser.add_argument("--val_file", type=str, required=True)
    parser.add_argument("--output_dir", type=str, default="./models/qwen-darija-checkpoints")
    parser.add_argument("--adapter_dir", type=str, default="./models/qwen-darija-lora")
    parser.add_argument("--epochs", type=int, default=1)
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--lr", type=float, default=2e-4)
    return parser.parse_args()

def main():
    args = parse_args()
    print(f"Chargement du tokenizer pour {args.model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print("Chargement et préparation des datasets...")
    dataset = load_dataset("json", data_files={"train": args.train_file, "validation": args.val_file})

    def tokenize_function(batch):
        texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in batch["messages"]]
        return tokenizer(texts, truncation=True, max_length=256)

    tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset["train"].column_names)

    print("Configuration du modèle quantifié (QLoRA)...")
    use_cuda = torch.cuda.is_available()
    bnb_config = None
    device_map = None

    if use_cuda:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        device_map = "auto"
        torch_dtype = torch.float16
    else:
        print("⚠️ Aucun GPU détecté, exécution sur CPU sans quantification (mode test/développement).")
        torch_dtype = torch.float32

    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        quantization_config=bnb_config,
        device_map=device_map,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True
    )
    model.config.use_cache = False

    if use_cuda:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    training_args = TrainingArguments(
        output_dir=args.output_dir,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=4 if use_cuda else 1,
        learning_rate=args.lr,
        weight_decay=0.01,
        warmup_steps=100,
        lr_scheduler_type="cosine",
        fp16=use_cuda,
        use_cpu=not use_cuda,
        gradient_checkpointing=use_cuda,
        optim="paged_adamw_8bit" if use_cuda else "adamw_torch",
        evaluation_strategy="steps",
        eval_steps=250,
        save_strategy="steps",
        save_steps=250,
        save_total_limit=2,
        logging_steps=50,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    )

    print("Lancement de l'entraînement...")
    trainer.train()

    print(f"Sauvegarde des adaptateurs LoRA finaux dans : {args.adapter_dir}")
    model.save_pretrained(args.adapter_dir)
    tokenizer.save_pretrained(args.adapter_dir)
    print("Entraînement et sauvegarde terminés avec succès !")

if __name__ == "__main__":
    main()
"""

with open(src_dir / "train.py", "w", encoding="utf-8") as f:
    f.write(train_script_content.strip())

print("src/train.py créé avec succès !")

src/train.py créé avec succès !


### Étape 4 : Création du script `src/push.py` pour Hugging Face

In [117]:
push_script_content = """# -*- coding: utf-8 -*-
import argparse
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

def push_to_hub(repo_name, adapter_dir):
    print(f"Connexion et envoi de l'adaptateur LoRA de {adapter_dir} vers Hugging Face Hub: {repo_name}...")
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
    config = PeftConfig.from_pretrained(adapter_dir)
    base_model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path)
    model = PeftModel.from_pretrained(base_model, adapter_dir)

    model.push_to_hub(repo_name)
    tokenizer.push_to_hub(repo_name)
    print("Adaptateur envoyé avec succès !")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--repo_name", type=str, required=True, help="Exemple: nom_utilisateur/qwen-darija-lora")
    parser.add_argument("--adapter_dir", type=str, default="./models/qwen-darija-lora")
    args = parser.parse_args()
    push_to_hub(args.repo_name, args.adapter_dir)
"""

with open(src_dir / "push.py", "w", encoding="utf-8") as f:
    f.write(push_script_content.strip())

print("src/push.py créé avec succès !")

src/push.py créé avec succès !


In [118]:
import subprocess
from pathlib import Path

# Chemin du projet
project_path = Path("/content/Moroccan_Darija_LLM")

def run_git_cmd(args, cwd=project_path):
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Erreur lors de l'exécution de {' '.join(args)}:")
        print(result.stderr)
    else:
        print(result.stdout.strip() or f"Succès : {' '.join(args)}")

# 1. Initialiser le dépôt local Git si ce n'est pas déjà fait
run_git_cmd(["git", "init"])

# 2. Configurer temporairement un nom et email d'auteur pour le commit
run_git_cmd(["git", "config", "user.name", "Herchan321"])
run_git_cmd(["git", "config", "user.email", "colab-developer@example.com"])

# 3. Ajouter les fichiers essentiels
run_git_cmd(["git", "add", "README.md", "requirements.txt", "src/"])

# 4. Effectuer le premier commit
run_git_cmd(["git", "commit", "-m", "Initial commit: robust pipeline & structure for Moroccan Darija LLM SFT"])

# 5. Définir la branche principale
run_git_cmd(["git", "branch", "-M", "main"])

# 6. Ajouter le remote origin (en écrasant s'il existe déjà)
run_git_cmd(["git", "remote", "remove", "origin"])
run_git_cmd(["git", "remote", "add", "origin", "https://github.com/Herchan321/Moroccan_Darija_LLM.git"])

print("\nStructure Git locale initialisée et prête à être poussée !")
print("Exécutez la cellule suivante pour vous authentifier et pousser vos commits sur GitHub.")

Initialized empty Git repository in /content/Moroccan_Darija_LLM/.git/
Succès : git config user.name Herchan321
Succès : git config user.email colab-developer@example.com
Succès : git add README.md requirements.txt src/
[master (root-commit) 05f8c38] Initial commit: robust pipeline & structure for Moroccan Darija LLM SFT
 4 files changed, 213 insertions(+)
 create mode 100644 README.md
 create mode 100644 requirements.txt
 create mode 100644 src/push.py
 create mode 100644 src/train.py
Succès : git branch -M main
Erreur lors de l'exécution de git remote remove origin:
error: No such remote: 'origin'

Succès : git remote add origin https://github.com/Herchan321/Moroccan_Darija_LLM.git

Structure Git locale initialisée et prête à être poussée !
Exécutez la cellule suivante pour vous authentifier et pousser vos commits sur GitHub.


```markdown
### Authentification et Push vers GitHub
Pour pousser vos commits sur un dépôt privé ou public de manière sécurisée en Colab sans exposer vos mots de passe, nous vous conseillons d'utiliser un **Personal Access Token (PAT)** GitHub disponible dans vos paramètres développeurs sur GitHub.

Une fois votre token généré, vous pouvez l'enregistrer dans les secrets de votre Notebook Colab sous la clé `GITHUB_TOKEN`, puis exécuter la cellule ci-dessous :
```

In [122]:
from google.colab import userdata
import subprocess

try:
    # Récupérer le token d'accès GitHub depuis vos secrets Colab
    github_token = userdata.get('GITHUB_TOKEN')
    repo_url = f"https://{github_token}@github.com/Herchan321/Moroccan_Darija_LLM.git"

    print("Envoi en cours vers GitHub...")
    result = subprocess.run(
        ["git", "push", "-u", repo_url, "main"],
        cwd="/content/Moroccan_Darija_LLM",
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("🚀 Projet poussé avec succès sur GitHub !")
        print("Lien de votre dépôt : https://github.com/Herchan321/Moroccan_Darija_LLM")
    else:
        print("Erreur lors du Push :\n", result.stderr)
except Exception as e:
    print("Erreur d'authentification ou Secret 'GITHUB_TOKEN' manquant :", e)

Envoi en cours vers GitHub...
🚀 Projet poussé avec succès sur GitHub !
Lien de votre dépôt : https://github.com/Herchan321/Moroccan_Darija_LLM
